# T12 · trigrama online en Colab

Carga el checkpoint causal existente, conecta el trigrama oficial y evalúa val completa. Ejecuta las celdas en orden. No entrena la GRU ni evalúa test.

La GRU conserva PyTorch 2.5.1; el decodificador de palabras usa otro entorno CPU con PyTorch 1.13.1. Primero se comprueba que recibir los mismos logits juntos o por bloques produce la misma frase.

La primera preparación descarga `languageModel.tar.gz` (14,1 GB comprimido) y compila el runtime. El grafo preparado y los resultados se guardan en Drive. Si Dryad rechaza la descarga, la celda 7 permite usar una copia de Drive.


In [2]:
#@title 1. Abrir en Colab y elegir el proyecto
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab
except ImportError as error:
    raise RuntimeError("Ejecuta este notebook en Colab.") from error

previous = globals().get("PROJECT")
COLAB_BASE = Path(globals().get("COLAB_BASE", Path.cwd().resolve()))
PROJECT = COLAB_BASE / "tic_proyecto_trigrama_colab"
candidates = ([Path(previous)] if previous is not None else []) + [
    COLAB_BASE / "tic_proyecto_online_colab", COLAB_BASE / "tic_proyecto_colab",
]
PREVIOUS_PROJECT = next((candidate for candidate in candidates
                         if candidate != PROJECT and (candidate / "data/speech_2023").is_dir()), None)
CHECK_REPORT = None
print("Proyecto:", PROJECT)


Proyecto: /content/tic_proyecto_trigrama_colab


## Copiar el código

El paquete contiene las fuentes de esta fase y sus hashes.


In [3]:
#@title 2. Copiar el código del proyecto (snapshot incluido)
import base64
import hashlib
import io
import json
from pathlib import PurePosixPath
from zipfile import ZipFile

SOURCE_SHA256 = '93ff5cbcd488540c0ba10aa002b6de9257e9f8b0a31d97540b04b1b8f2d79d9a'
SOURCE_B64 = (
    '...'
)

SOURCE_BYTES = base64.b64decode(SOURCE_B64)
if hashlib.sha256(SOURCE_BYTES).hexdigest() != SOURCE_SHA256:
    raise ValueError("El paquete de código no supera su SHA-256.")
PROJECT.mkdir(parents=True, exist_ok=True)
with ZipFile(io.BytesIO(SOURCE_BYTES)) as archive:
    for member in archive.infolist():
        relative = PurePosixPath(member.filename)
        if relative.is_absolute() or ".." in relative.parts or member.is_dir():
            raise ValueError(f"Ruta inesperada en el paquete: {member.filename}")
        destination = PROJECT.joinpath(*relative.parts)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(member))
SOURCE_MANIFEST = json.loads((PROJECT / "source_snapshot.json").read_text())
for name, expected in SOURCE_MANIFEST["files"].items():
    if hashlib.sha256((PROJECT / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Código distinto del snapshot: {name}")
print("Código preparado:", len(SOURCE_MANIFEST["files"]), "archivos")
print("Commit base:", SOURCE_MANIFEST["git_commit_base"])
print("SHA-256 del paquete:", SOURCE_SHA256)


Código preparado: 62 archivos
Commit base: 3ad8ac389c8932263bcc63048bab655bee2ed849
SHA-256 del paquete: 93ff5cbcd488540c0ba10aa002b6de9257e9f8b0a31d97540b04b1b8f2d79d9a


## Instalar el entorno de la GRU


In [4]:
#@title 3. Instalar el entorno del proyecto
import shutil

subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "uv==0.9.4"], check=True)
UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("No se encontró uv después de instalarlo.")
PYTHON = PROJECT / ".venv" / "bin" / "python"
if not PYTHON.is_file():
    subprocess.run([UV, "venv", "--python", "3.12.11", "--seed", str(PROJECT / ".venv")], check=True)
subprocess.run([UV, "pip", "install", "--python", str(PYTHON), "-r", "requirements.txt"], cwd=PROJECT, check=True)
subprocess.run([UV, "pip", "check", "--python", str(PYTHON)], cwd=PROJECT, check=True)

def ejecutar(argumentos: list[str]) -> None:
    """Muestra stdout/stderr en Colab y conserva el motivo real si falla."""
    from collections import deque
    import signal

    recent = deque(maxlen=25)
    process = subprocess.Popen([str(PYTHON), *argumentos], cwd=PROJECT,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding="utf-8", errors="replace", bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            recent.append(line)
        returncode = process.wait()
    finally:
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
        process.stdout.close()
    if returncode:
        detail = "".join(recent).strip() or "El proceso terminó sin producir salida."
        raise RuntimeError(f"Falló {argumentos[0]} (código {returncode}).\n{detail}")

print("Entorno aislado preparado. No hace falta reiniciar el kernel.")


Entorno aislado preparado. No hace falta reiniciar el kernel.


## Elegir pesos y configuración


In [5]:
#@title 4. Conectar Drive y seleccionar el checkpoint existente
CARPETA_DRIVE = "speech_bci" #@param {type:"string"}
CHECKPOINT_RELATIVO = "speech_online_baseline/jaovk23p/mejor_val.pt" #@param {type:"string"}
DISPOSITIVO = "cuda" #@param ["cpu", "cuda"]

from datetime import datetime
from zoneinfo import ZoneInfo
from google.colab import drive
from uuid import uuid4
import json

relative = Path(CARPETA_DRIVE.strip())
checkpoint_relative = Path(CHECKPOINT_RELATIVO.strip())
if (relative.is_absolute() or ".." in relative.parts or not CARPETA_DRIVE.strip()
        or not CHECKPOINT_RELATIVO.strip() or checkpoint_relative.is_absolute()
        or ".." in checkpoint_relative.parts):
    raise ValueError("Usa rutas relativas a Mi unidad y a checkpoints, respectivamente.")
CHECK_REPORT = None
DRIVE_MOUNT = COLAB_BASE / "drive"
drive.mount(str(DRIVE_MOUNT))
DRIVE_ROOT = DRIVE_MOUNT / "MyDrive" / relative
CHECKPOINT_PATH = DRIVE_ROOT / "checkpoints" / checkpoint_relative
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(CHECKPOINT_PATH)
experiment_id = datetime.now(ZoneInfo("Europe/Madrid")).strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8]
RESULTS = DRIVE_ROOT / "experimentos" / ("trigrama_t12_" + experiment_id)
RESULTS.mkdir(parents=True, exist_ok=False)
(RESULTS / "fuentes.zip").write_bytes(SOURCE_BYTES)
(RESULTS / "source_snapshot.json").write_text(json.dumps(SOURCE_MANIFEST, indent=2), encoding="utf-8")
CONFIG = "configs/entrenos_online/speech_colab.yaml"
EVALUATION_CONFIG = "configs/entrenos_online/evaluacion_trigrama_colab.yaml"
LANGUAGE_CONFIG = "configs/entrenos_online/trigrama_colab.yaml"
configure = """
from pathlib import Path
import json, sys, torch, yaml
from src.inference.checkpoints import read_speech_checkpoint
from src.data.speech_dataset import file_sha256
state = read_speech_checkpoint(Path(sys.argv[1]))
if state['format_version'] != 3:
    raise ValueError('Selecciona un checkpoint entrenado con preprocesamiento causal.')
if sys.argv[5] == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('Selecciona una GPU en el entorno de Colab o elige cpu.')
cfg = state['config']
Path(sys.argv[2]).write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
evaluation = yaml.safe_load(Path('configs/entrenos_online/speech_online_evaluacion.yaml').read_text())
evaluation.update(split='val', dispositivo=sys.argv[5], permitir_tf32=False)
Path(sys.argv[3]).write_text(yaml.safe_dump(evaluation, sort_keys=False), encoding='utf-8')
language = yaml.safe_load(Path('configs/entrenos_online/speech_trigrama.yaml').read_text())
language['modelo'] = str(Path(sys.argv[6]) / 'modelos_lenguaje' / 'speech_trigram')
Path(sys.argv[4]).write_text(yaml.safe_dump(language, sort_keys=False), encoding='utf-8')
print(json.dumps({'config':cfg, 'language':language, 'evaluation':evaluation,
                  'checkpoint_sha256':file_sha256(Path(sys.argv[1]))}))
"""
settings = json.loads(subprocess.check_output([str(PYTHON), "-c", configure, str(CHECKPOINT_PATH), CONFIG,
                                             EVALUATION_CONFIG, LANGUAGE_CONFIG, DISPOSITIVO, str(DRIVE_ROOT)],
                                            cwd=PROJECT, text=True))
cfg = settings["config"]
language_cfg = settings["language"]
CHECKPOINT_SHA256 = settings["checkpoint_sha256"]
for name in (CONFIG, EVALUATION_CONFIG, LANGUAGE_CONFIG):
    shutil.copyfile(PROJECT / name, RESULTS / Path(name).name)
shutil.copyfile(PROJECT / cfg["protocolo"]["manifiesto"], RESULTS / "particion_t12_v1.json")
for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[variable] = str(settings["evaluation"]["hilos_cpu"])
data_link = PROJECT / "data" / "speech_2023"
if PREVIOUS_PROJECT is not None and not data_link.exists():
    data_link.parent.mkdir(parents=True, exist_ok=True)
    data_link.symlink_to(PREVIOUS_PROJECT / "data" / "speech_2023", target_is_directory=True)
    print("Se reutilizan los datos de la sesión anterior.")
print("Checkpoint:", CHECKPOINT_PATH)
print("SHA-256:", CHECKPOINT_SHA256)
print("Modelo:", cfg["modelo"])
print("Semilla de entrenamiento:", cfg["experimento"]["seed"], "· Evaluación:", DISPOSITIVO)
print("Resultados:", RESULTS)


Mounted at /content/drive
Checkpoint: /content/drive/MyDrive/speech_bci/checkpoints/speech_online_baseline/jaovk23p/mejor_val.pt
SHA-256: 853fa9c706c86ca75bcdba74208a4749389d24ca69c7309d535eb2e8697a1ebe
Modelo: {'nombre': 'speech_gru', 'entrada': 256, 'capa_dia': 256, 'capas': 5, 'unidades': 512, 'bidireccional': False, 'kernel': 14, 'stride': 4, 'dropout_entrada': 0.2, 'dropout_gru': 0.4}
Semilla de entrenamiento: 0 · Evaluación: cuda
Resultados: /content/drive/MyDrive/speech_bci/experimentos/trigrama_t12_20261101_210724_101948c3


## Datos de validación

Reutiliza las señales preparadas si siguen en esta sesión. En una sesión nueva, prepara el mismo protocolo fijo.


In [6]:
#@title 5. Obtener y extraer competitionData
ORIGEN_DATOS = "Drive" #@param ["Drive", "Dryad"]
ARCHIVO_DATOS_DRIVE = "" #@param {type:"string"}


def buscar_archivos_drive(root: Path) -> tuple[list[Path], bool]:
    """Busca el tar de competitionData por nombre, sin leer su contenido."""
    import time

    matches = []
    started = time.monotonic()
    visited = 0
    incomplete = False
    errors = []
    for current, directories, filenames in os.walk(root, onerror=errors.append, followlinks=False):
        visited += 1
        folder = Path(current)
        if len(folder.relative_to(root).parts) >= 5:
            incomplete = incomplete or bool(directories)
            directories.clear()
        else:
            directories[:] = sorted(name for name in directories if not name.startswith('.'))
        for name in sorted(filenames):
            if name.casefold().startswith('competitiondata') and name.casefold().endswith(('.tar.gz', '.tgz')):
                matches.append(folder / name)
        if visited >= 200 or time.monotonic() - started >= 15:
            incomplete = True
            break
    return matches, incomplete or bool(errors)


manifest = json.loads((PROJECT / "docs/datasets/speech_2023/manifest.json").read_text())
expected = next(item for item in manifest["files"] if item["name"] == "competitionData.tar.gz")
destination = PROJECT / "data" / "speech_2023" / expected["name"]
destination.parent.mkdir(parents=True, exist_ok=True)
my_drive = DRIVE_MOUNT / "MyDrive"
print("Tamaño esperado:", expected["size_bytes"], "bytes")
print("Espacio libre en Colab:", round(shutil.disk_usage(PROJECT).free / (1 << 30), 1), "GiB")
if ORIGEN_DATOS not in ("Drive", "Dryad"):
    raise ValueError("Selecciona Drive o Dryad en ORIGEN_DATOS.")

local_valid = False
if destination.is_file():
    try:
        ejecutar(["scripts/get_data.py", "--dataset", "speech_2023", "--files", expected["name"], "--verify-only"])
        local_valid = True
        print("Se reutiliza la copia local verificada.")
    except RuntimeError:
        print("La copia local no es válida; solo se sustituirá por un archivo verificado.")

if not local_valid and ORIGEN_DATOS == "Drive":
    if not my_drive.is_dir():
        raise RuntimeError("Drive no está disponible. Ejecuta la celda 4 para montarlo antes de continuar.")
    entered = ARCHIVO_DATOS_DRIVE.strip().strip("\"'")
    if entered:
        selected = Path(entered)
        if ".." in selected.parts:
            raise ValueError("Indica la ruta del archivo sin componentes '..'.")
        if selected.is_absolute():
            if not selected.is_relative_to(DRIVE_MOUNT):
                raise ValueError(f"La ruta debe estar dentro del Drive montado en {DRIVE_MOUNT}.")
            source = selected
        elif selected.parts and selected.parts[0] in ("MyDrive", "Shareddrives"):
            source = DRIVE_MOUNT / selected
        else:
            source = my_drive / selected
        if not source.is_file():
            raise FileNotFoundError(
                f"Colab no ve un archivo en: {source}\n"
                "En el panel Archivos, localízalo dentro de drive y usa 'Copiar ruta'. "
                "Comprueba que Drive está montado con la cuenta donde subiste el archivo. "
                "No se ha intentado descargar de Dryad."
            )
    else:
        candidates = [DRIVE_ROOT / "datos" / expected["name"], DRIVE_ROOT / expected["name"], my_drive / expected["name"]]
        source = next((candidate for candidate in candidates if candidate.is_file()), None)
        if source is None:
            print("Buscando competitionData en Mi unidad (hasta 5 niveles y 200 carpetas)...", flush=True)
            matches, incomplete = buscar_archivos_drive(my_drive)
            if incomplete:
                print("Búsqueda limitada: puede haber otras carpetas. Puedes indicar la ruta exacta.")
            complete = [candidate for candidate in matches if candidate.stat().st_size == expected["size_bytes"]]
            choices = complete or matches
            if len(choices) == 1:
                source = choices[0]
            else:
                if choices:
                    detail = "Se han encontrado varios archivos:\n" + "\n".join(str(candidate) for candidate in choices)
                else:
                    detail = "No se encontró el archivo. Rutas iniciales comprobadas:\n" + "\n".join(str(candidate) for candidate in candidates)
                raise RuntimeError(
                    f"{detail}\n\nEn el panel Archivos de Colab, localiza el archivo dentro de drive, "
                    "pulsa 'Copiar ruta' y pégala en ARCHIVO_DATOS_DRIVE. "
                    "Comprueba la cuenta de Drive si no aparece en el panel. "
                    "Vuelve a ejecutar solo esta celda. No se ha intentado descargar de Dryad."
                )
    print("Archivo encontrado en Drive:", source, flush=True)
    size = source.stat().st_size
    if size != expected["size_bytes"]:
        raise ValueError(f"El archivo de Drive tiene {size} bytes; se esperaban {expected['size_bytes']}. Comprueba que terminó la subida y que es el archivo original del 1 de septiembre de 2023.")
    temporary = destination.with_name(destination.name + ".drive.part")
    print("Copiando al disco de Colab; después se comprobará el SHA-256...", flush=True)
    shutil.copyfile(source, temporary)
    verification = """
from pathlib import Path
import sys
from scripts.get_data import leer_manifiesto, verificar
source = next(item for item in leer_manifiesto('speech_2023')['files'] if item['name'] == 'competitionData.tar.gz')
verificar(Path(sys.argv[1]), source)
print('Copia de Drive: tamaño y SHA-256 oficiales verificados.')
"""
    ejecutar(["-c", verification, str(temporary)])
    temporary.replace(destination)
elif not local_valid:
    ejecutar(["scripts/get_data.py", "--dataset", "speech_2023", "--files", expected["name"]])

extract = """
from pathlib import Path
import json
import tarfile
import yaml
cfg = yaml.safe_load(Path('configs/entrenos_online/speech_colab.yaml').read_text())
manifest = json.loads(Path('docs/datasets/speech_2023/manifest.json').read_text())
digest = next(item['sha256'] for item in manifest['files'] if item['name'] == 'competitionData.tar.gz')
marker = Path('data/speech_2023/.extraccion_verificada')
expected = [Path(cfg['datos']['raiz']) / split / (session + '.mat') for split in ['train', 'test'] for session in cfg['datos']['sesiones']]
if not marker.exists() or marker.read_text().strip() != digest or not all(path.is_file() for path in expected):
    with tarfile.open('data/speech_2023/competitionData.tar.gz') as archive:
        archive.extractall('data/speech_2023', filter='data')
    marker.write_text(digest)
print('Extraídas las señales originales. Las 19 sesiones seleccionadas están disponibles.')
"""
ejecutar(["-c", extract])


Tamaño esperado: 3669638707 bytes
Espacio libre en Colab: 187.4 GiB
Archivo encontrado en Drive: /content/drive/MyDrive/speech_bci/datos/competitionData.tar.gz
Copiando al disco de Colab; después se comprobará el SHA-256...
Copia de Drive: tamaño y SHA-256 oficiales verificados.
Extraídas las señales originales. Las 19 sesiones seleccionadas están disponibles.


In [7]:
#@title 6. Preparar recursos, fonemas y señales

ejecutar(["scripts/get_phoneme_resources.py", "--config", CONFIG])
labels = PROJECT / cfg["etiquetas"]["salida"] / "manifest.json"
signals = PROJECT / cfg["datos"]["preparados"] / "manifest.json"
if not labels.is_file():
    ejecutar(["scripts/prepare_phoneme_labels.py", "--config", CONFIG])
if not signals.is_file():
    ejecutar(["scripts/prepare_speech_data.py", "--config", CONFIG])
verify = """
from pathlib import Path
import json
import yaml
from src.data.speech_dataset import SpeechDataset
from src.data.speech_protocol import read_speech_protocol
cfg = yaml.safe_load(Path('configs/entrenos_online/speech_colab.yaml').read_text())
protocol = read_speech_protocol(cfg)
train = SpeechDataset(cfg, 'train')
val = SpeechDataset(cfg, 'val')
if (len(train), len(val), len(train.sessions)) != (5180, 1460, 19):
    raise ValueError('Los datos no corresponden al protocolo T12 fijado.')
if protocol['totals']['test'] != 680:
    raise ValueError('El test oficial ha cambiado.')
print(json.dumps({'train': len(train), 'val': len(val), 'test_reservado': 680,
                  'sesiones': len(train.sessions), 'normalizacion': 'solo train',
                  'hashes': 'verificados', 'protocolo_sha256': cfg['protocolo']['sha256']}, indent=2))
"""
ejecutar(["-c", verify])
shutil.copyfile(signals, RESULTS / "manifiesto_preparacion.json")
shutil.copyfile(labels, RESULTS / "manifiesto_fonemas.json")


Verificado y extraído: corpora/cmudict.zip
Verificado y extraído: taggers/averaged_perceptron_tagger.zip
Verificado y extraído: taggers/averaged_perceptron_tagger_eng.zip
Recursos de fonemas preparados y verificados.
/content/tic_proyecto_trigrama_colab/.venv/lib/python3.12/site-packages/g2p_en/g2p.py:156: SyntaxWarning: invalid escape sequence '\-'
  text = re.sub("[^ a-z'.,?!\-]", "", text)
Etiquetas preparadas: t12.2022.04.28
Etiquetas preparadas: t12.2022.05.05
Etiquetas preparadas: t12.2022.05.17
Etiquetas preparadas: t12.2022.05.19
Etiquetas preparadas: t12.2022.05.24
Etiquetas preparadas: t12.2022.05.26
Etiquetas preparadas: t12.2022.06.02
Etiquetas preparadas: t12.2022.06.07
Etiquetas preparadas: t12.2022.06.14
Etiquetas preparadas: t12.2022.06.16
Etiquetas preparadas: t12.2022.06.21
Etiquetas preparadas: t12.2022.06.28
Etiquetas preparadas: t12.2022.07.05
Etiquetas preparadas: t12.2022.07.14
Etiquetas preparadas: t12.2022.07.21
Etiquetas preparadas: t12.2022.07.27
Etiquetas pr

PosixPath('/content/drive/MyDrive/speech_bci/experimentos/trigrama_t12_20261101_210724_101948c3/manifiesto_fonemas.json')

## Preparar el trigrama

La compilación tarda y se realiza en la máquina de Colab. El entorno de la GRU se conserva. El grafo ya preparado en Drive se verifica y se reutiliza.


In [8]:
#@title 7. Obtener el grafo y compilar el runtime CPU
ORIGEN_LM = "Drive" #@param ["Dryad", "Drive"]
ARCHIVO_LM_DRIVE = "" #@param {type:"string"}

if ORIGEN_LM not in ("Drive", "Dryad"):
    raise ValueError("Selecciona Drive o Dryad en ORIGEN_LM.")
model_root = Path(language_cfg["modelo"])
archive_name = "languageModel.tar.gz"
destination = PROJECT / "data" / "speech_2023" / archive_name
if not (model_root / "manifest.json").is_file() and not destination.is_file():
    if ORIGEN_LM == "Drive":
        entered = ARCHIVO_LM_DRIVE.strip().strip("\"'")
        if entered:
            selected = Path(entered)
            if ".." in selected.parts:
                raise ValueError("Indica una ruta sin '..'.")
            if selected.is_absolute():
                if not selected.is_relative_to(DRIVE_MOUNT):
                    raise ValueError("El archivo debe estar en el Drive montado.")
                source = selected
            elif selected.parts and selected.parts[0] in ("MyDrive", "Shareddrives"):
                source = DRIVE_MOUNT / selected
            else:
                source = DRIVE_MOUNT / "MyDrive" / selected
        else:
            source = next((candidate for candidate in (DRIVE_ROOT / "datos" / archive_name,
                           DRIVE_ROOT / archive_name, DRIVE_MOUNT / "MyDrive" / archive_name)
                           if candidate.is_file()), None)
        if source is None or not source.is_file():
            raise FileNotFoundError("Indica la ruta de languageModel.tar.gz en ARCHIVO_LM_DRIVE.")
        destination.parent.mkdir(parents=True, exist_ok=True)
        expected = next(item for item in json.loads((PROJECT / "docs/datasets/speech_2023/manifest.json").read_text())["files"]
                        if item["name"] == archive_name)
        if source.stat().st_size != expected["size_bytes"]:
            raise ValueError("El archivo de Drive no tiene el tamaño oficial.")
        if shutil.disk_usage(PROJECT).free < source.stat().st_size + 5 * (1 << 30):
            raise RuntimeError("Falta espacio para copiar el archivo al disco de Colab.")
        temporary = destination.with_name(archive_name + ".drive.part")
        print("Copiando el archivo de Drive al disco de Colab...", flush=True)
        shutil.copyfile(source, temporary)
        verify = "from scripts.get_data import leer_manifiesto, verificar; from pathlib import Path; import sys; verificar(Path(sys.argv[1]), next(x for x in leer_manifiesto('speech_2023')['files'] if x['name']=='languageModel.tar.gz'))"
        ejecutar(["-c", verify, str(temporary)])
        temporary.rename(destination)
    else:
        ejecutar(["scripts/get_data.py", "--dataset", "speech_2023", "--files", archive_name])
ejecutar(["scripts/prepare_speech_trigram.py", "--config", LANGUAGE_CONFIG, "--prepare-model", "--build-runtime"])
runtime_python = PROJECT / language_cfg["python_runtime"]
shutil.copyfile(runtime_python.parent.parent / "runtime_build.json", RESULTS / "runtime_build.json")
shutil.copyfile(model_root / "manifest.json", RESULTS / "modelo_lenguaje.json")
print("Trigrama y runtime preparados.")


Cloning into '/content/tic_proyecto_trigrama_colab/data/language_models/speechBCI'...
HEAD is now at 522a960 Update boost link
Using CPython 3.10.15
Creating virtual environment with seed packages at: data/language_models/runtime
 + packaging==26.3
 + pip==26.2.2
 + setuptools==84.2.1
 + wheel==0.48.0
Activate with: source data/language_models/runtime/bin/activate
Using Python 3.10.15 environment at: data/language_models/runtime
Resolved 7 packages in 715ms
Prepared 7 packages in 3.55s
Uninstalled 3 packages in 16ms
Installed 7 packages in 167ms
 + cmake==3.27.9
 + numpy==1.24.4
 - pip==26.2.2
 + pip==24.3.1
 - setuptools==84.2.1
 + setuptools==68.2.2
 + torch==1.13.1+cpu
 + typing-extensions==4.15.0
 - wheel==0.48.0
 + wheel==0.41.3
running install
/content/tic_proyecto_trigrama_colab/data/language_models/runtime/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        **************************************

## Registro en W&B


In [9]:
#@title 8. Cargar la credencial de W&B sin mostrarla
import getpass

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.utils.env import cargar_clave_wandb

if not cargar_clave_wandb(PROJECT / ".env", DRIVE_ROOT / ".env"):
    from google.colab import userdata
    try:
        api_key = userdata.get("WANDB_API_KEY")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
        api_key = None
    if not api_key or not api_key.strip():
        api_key = getpass.getpass("API key de W&B (entrada oculta): ")
    if not api_key or not api_key.strip():
        raise ValueError("Falta la credencial de W&B.")
    os.environ["WANDB_API_KEY"] = api_key.strip()
    del api_key
print("Credencial cargada. W&B mostrará el enlace del run al iniciar la evaluación.")


Credencial cargada. W&B mostrará el enlace del run al iniciar la evaluación.


## Comprobar antes de evaluar

Usa una frase de val y compara el WFST con los mismos logits agrupados o por bloques. No representa una métrica del corpus.


In [10]:
#@title 9. Comprobar el estado incremental del trigrama
def evaluar_trigrama(destino: Path, *, solo_comprobar: bool = False) -> None:
    """Lanza la evaluación y muestra el log del trabajador si falla."""
    argumentos = ["scripts/evaluate_speech_trigram.py", "--checkpoint", str(CHECKPOINT_PATH),
                  "--evaluation-config", EVALUATION_CONFIG, "--language-config", LANGUAGE_CONFIG,
                  "--split", "val", "--output", str(destino)]
    if solo_comprobar:
        argumentos.append("--check-streaming")
    try:
        ejecutar(argumentos)
    except RuntimeError:
        log = destino / "trigram_worker.log"
        if log.is_file():
            print("Últimas líneas del trabajador:")
            print("\n".join(log.read_text(errors="replace").splitlines()[-40:]))
        raise

CHECK_REPORT = None
CHECK_OUTPUT = RESULTS / ("comprobacion_" + uuid4().hex[:8])
evaluar_trigrama(CHECK_OUTPUT, solo_comprobar=True)
report = json.loads((CHECK_OUTPUT / "comprobacion_incremental.json").read_text())
if not report["verified"] or report["checkpoint_sha256"] != CHECKPOINT_SHA256:
    raise RuntimeError("La comprobación incremental no pasó con el checkpoint seleccionado.")
CHECK_REPORT = report
print(json.dumps(CHECK_REPORT["profiles"], indent=2))


Comprobación incremental correcta: /content/drive/MyDrive/speech_bci/experimentos/trigrama_t12_20261101_210724_101948c3/comprobacion_399ade10
{
  "5": {
    "matches": true,
    "complete": "the giant redwood city in the rising sun",
    "streamed": "the giant redwood city in the rising sun",
    "frames": 121
  }
}


In [11]:
#@title Diagnosticar la carga del trigrama sin entrenar ni recompilar
from pathlib import Path
import json
import os
import signal
import subprocess
import tempfile
import time
from uuid import uuid4

try:
    import google.colab
except ImportError as error:
    raise RuntimeError("Esta celda se ejecuta en la sesión de Colab del notebook 13.") from error

if not all(name in globals() for name in ("PROJECT", "RESULTS", "language_cfg")):
    raise RuntimeError("Se necesitan las variables de las celdas 1 a 4 del notebook 13.")

project = Path(PROJECT)
python_runtime = project / language_cfg["python_runtime"]
graph = Path(language_cfg["modelo"]) / "TLG.fst"
worker = project / "scripts/trigram_worker.py"

def memoria_disponible() -> int:
    """Consulta la RAM disponible del sistema, en bytes."""
    for line in Path("/proc/meminfo").read_text().splitlines():
        if line.startswith("MemAvailable:"):
            return int(line.split()[1]) * 1024
    raise RuntimeError("No se pudo consultar MemAvailable.")

def eventos_memoria() -> str:
    """Lee los contadores del cgroup si el entorno permite consultarlos."""
    try:
        return Path("/sys/fs/cgroup/memory.events").read_text()
    except OSError:
        return "no disponible"

report = {"graph_bytes": graph.stat().st_size, "memory_events_before": eventos_memoria(),
          "cgroup": Path("/proc/self/cgroup").read_text(), "peak_worker_rss_bytes": 0,
          "stopped_by_probe": None}
# Umbrales de esta comprobación; no cambian la búsqueda ni los parámetros del modelo.
max_rss = 8 * (1 << 30)
min_available = 2 * (1 << 30)
max_seconds = 90
environment = {**os.environ, "CUDA_VISIBLE_DEVICES": ""}
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    environment[name] = str(language_cfg["hilos_cpu"])
request = {"op": "init", "model": str(graph.parent), "options": language_cfg["busqueda"],
           "threads": language_cfg["hilos_cpu"]}

print(f"Grafo: {report['graph_bytes'] / (1 << 30):.2f} GiB en disco", flush=True)
print("Prueba del runtime existente; parada a 8 GiB de RSS, menos de 2 GiB disponibles o 90 s.", flush=True)
with tempfile.TemporaryDirectory(prefix="trigram_load_probe_") as folder:
    stdout_path, stderr_path = Path(folder) / "stdout.log", Path(folder) / "stderr.log"
    with stdout_path.open("w") as out, stderr_path.open("w") as err:
        process = subprocess.Popen([str(python_runtime), "-u", "-X", "faulthandler", str(worker)],
                                   cwd=project, stdin=subprocess.PIPE, stdout=out, stderr=err,
                                   text=True, env=environment)
        started = last_print = time.monotonic()
        try:
            try:
                process.stdin.write(json.dumps(request) + "\n")
                process.stdin.close()
            except BrokenPipeError:
                pass
            while process.poll() is None:
                rss = 0
                try:
                    for line in Path(f"/proc/{process.pid}/status").read_text().splitlines():
                        if line.startswith("VmRSS:"):
                            rss = int(line.split()[1]) * 1024
                except FileNotFoundError:
                    continue
                report["peak_worker_rss_bytes"] = max(report["peak_worker_rss_bytes"], rss)
                available = memoria_disponible()
                now = time.monotonic()
                if rss >= max_rss or available < min_available:
                    report["stopped_by_probe"] = "límite preventivo de memoria de esta prueba"
                elif now - started >= max_seconds:
                    report["stopped_by_probe"] = "límite de tiempo de esta prueba"
                if report["stopped_by_probe"]:
                    process.terminate()
                    break
                if now - last_print >= 5:
                    print(f"RSS trabajador: {rss / (1 << 30):.2f} GiB; disponible: {available / (1 << 30):.2f} GiB", flush=True)
                    last_print = now
                time.sleep(0.1)
        finally:
            if not process.stdin.closed:
                try:
                    process.stdin.close()
                except BrokenPipeError:
                    pass
            if process.poll() is None:
                process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        report["returncode"] = process.returncode
        report["elapsed_seconds"] = time.monotonic() - started
    report["stdout"] = stdout_path.read_text(errors="replace")[-8192:]
    report["stderr"] = stderr_path.read_text(errors="replace")[-8192:]

report["memory_events_after"] = eventos_memoria()
if report["returncode"] < 0:
    try:
        report["signal"] = signal.Signals(-report["returncode"]).name
    except ValueError:
        report["signal"] = str(-report["returncode"])
report_path = Path(RESULTS) / ("diagnostico_carga_" + uuid4().hex[:8] + ".json")
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(json.dumps(report, ensure_ascii=False, indent=2))
print("Diagnóstico guardado:", report_path)


Grafo: 26.50 GiB en disco
Prueba del runtime existente; parada a 8 GiB de RSS, menos de 2 GiB disponibles o 90 s.
RSS trabajador: 0.93 GiB; disponible: 80.50 GiB
RSS trabajador: 2.16 GiB; disponible: 79.26 GiB
RSS trabajador: 3.44 GiB; disponible: 78.00 GiB
RSS trabajador: 4.72 GiB; disponible: 76.72 GiB
RSS trabajador: 6.01 GiB; disponible: 75.41 GiB
RSS trabajador: 7.33 GiB; disponible: 74.08 GiB
{
  "graph_bytes": 28458450578,
  "memory_events_before": "low 0\nhigh 0\nmax 0\noom 0\noom_kill 0\noom_group_kill 0\n",
  "cgroup": "0::/../../jupyter-children\n",
  "peak_worker_rss_bytes": 8594751488,
  "stopped_by_probe": "límite preventivo de memoria de esta prueba",
  "returncode": -15,
  "elapsed_seconds": 33.24799556800008,
  "stdout": "{\"ok\": true, \"python\": \"3.10.15\", \"torch\": \"1.13.1+cpu\", \"numpy\": \"1.24.4\"}\n",
  "stderr": "WARNING: Logging before InitGoogleLogging() is written to STDERR\nI1101 20:32:26.044090 28974 brain_speech_decoder.h:52] Reading fst /content/dr

## Evaluar val completa

Registra WER, PER, parciales y tiempos. Las llegadas se simulan sin red; los parciales de palabras pueden revisarse con nuevos bloques. Test sigue reservado.


In [12]:
#@title 10. Evaluar el trigrama en val
if not globals().get("CHECK_REPORT") or not CHECK_REPORT.get("verified"):
    raise RuntimeError("Ejecuta primero la comprobación de la celda 9.")
evaluation_output = RESULTS / ("validacion_" + uuid4().hex[:8])
evaluar_trigrama(evaluation_output)
evaluation = json.loads((evaluation_output / "metricas.json").read_text())
if (evaluation["optimizer_steps"] != 0 or evaluation["test_evaluated"]
        or evaluation["checkpoint_sha256"] != CHECKPOINT_SHA256):
    raise RuntimeError("La evaluación no corresponde al checkpoint y al protocolo seleccionados.")
for depth, values in evaluation["global"].items():
    print("Salida", depth, "· WER", f"{values['wer']:.2%}",
          f"({values['errores_palabras']}/{values['palabras']})", "· PER", f"{values['per']:.2%}",
          "· Frases", values["frases"], "· RTF", evaluation["timing"][depth]["rtf"])
print("Checkpoint SHA-256:", evaluation["checkpoint_sha256"])
print("Resultados:", evaluation_output)


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: aaurre (aaurre-deusto). Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.18.7
wandb: Run data is saved locally in runs/tracking/wandb/run-20261101_203841-e3gy3ugk
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run speech_online_baseline_seed0_colab_trigrama_val
wandb: ⭐️ View project at https://wandb.ai/aaurre-deusto/tiiac-2026
wandb: 🚀 View run at https://wandb.ai/aaurre-deusto/tiiac-2026/runs/e3gy3ugk
wandb: - 0.004 MB of 0.004 MB uploaded
wandb: \ 0.004 MB of 0.004 MB uploaded
wandb:                                                                                
wandb: 
wandb: Run history:
wandb: trigrama/tiempo/capa_5/first_partial_input/observations ▁
wandb:       trigrama/tiempo/capa_5/first_partial_input/p50_ms ▁
wandb:       trigrama/tiempo/capa_5/first_partial_input/p95_ms ▁
wandb: tri

In [13]:
#@title Preparar el trigrama compacto (CPU; sin entrenar)
# Copia esta celda completa en la sesión del notebook 13 que ya tiene el grafo.
from pathlib import Path, PurePosixPath
import base64
import copy
import hashlib
import io
import json
import os
import subprocess
from uuid import uuid4
from zipfile import ZipFile
import google.colab
import yaml

if not all(name in globals() for name in ("PROJECT", "RESULTS", "language_cfg", "PYTHON")):
    raise RuntimeError("Ejecuta esta celda en la sesión del notebook 13 ya preparada.")
previous_project = Path(PROJECT)
spec_compact = copy.deepcopy(language_cfg)
source_model = previous_project / language_cfg["modelo"]
source_runtime = previous_project / language_cfg["python_runtime"]
source_tree = previous_project / language_cfg["preparacion"]["fuentes"]
for required in (source_model / "manifest.json", source_model / "TLG.fst", source_runtime, source_tree):
    if not required.exists():
        raise FileNotFoundError(f"Falta un recurso de la sesión anterior: {required}")

SOURCE_SHA256 = 'd67440302a02eb27a6055ef6db0247c714d99f37ed4ef107863463bf9f0bc5e1'
SOURCE_B64 = (
    'UEsDBBQAAAAIAAAAIQComcqMFhIAAMI1AAAhAAAAc2NyaXB0cy9jb21wYWN0X3NwZWVjaF90cmlncmFtLnB5pVvbcttIkn3nV9RgHwbspqBLj3u6abMj3Lbc'
    '411fFJbtmQitAlEkilK1QACNiyyOwx/TH7APG/02r/qxPZlVAAoX0vaOHBYJoCorMyvz5AUlz/OepMmtVnmphIpFmeurXG6kSNd6pWUspMDzonxWlELGOlEy'
    'SsVWrNJNlldqKUVRiZXMr6TYyCyYTF5IPEtuVV7o+z8SURWYf/aO7ok3j1+CRCmF/ySN5XL6UCRVsgIFZSlg+RTL6wTLqmQSg+b9779VuBaREjH93qSBeJEW'
    'IlNFWsxEcf8/m2Ua48ZWyHyFT/qOxQqV30rxOlMJGA8mpzE9vta3KQiLSBerVGSVAtVVrlYqnxMTuYqqFa+T4F6hIxnRusxbHtPMUuVpMPE8bzJZ5+lGhOG6'
    'KqtchaHQmyzNoaIkSUtZanAwmdT38qtM5oWqr38t0qT+nhaGUibL61gvazJnuKyHFNdVqePmSl9BPc1VtczyFMwWzZ2tpViqTbbWsapJvlX0KfPtUw2RyzTf'
    '1lNKvVF2zjbTyVU943GybWTYyk08mbx5/fqtWDB3PmQH9TCcBjn2Ir5V/jSAmFBScXF8OQEfAQkVaNqL0j/CZpW5TxSmU6u+YpXrrMS4XNHMsMiUWl2H1gRr'
    'NpaVjqMwrxLic4bNu1JFaQnkK9BfK6y6UsEaJmrn5DDTENfhNb6ofIYbsCMsASPN5BJ84+FkMonUWshbqWO+t4F55Vt/Kg5+Ejop5xOBH+z262WpVaLES7V5'
    'XA+GZcTiOsWSL3RS3ZFZLbelIpvUCVtgBVPPxZN3Tx8HZDFEa53mgnwI1I0SvUPavkMsDDFSj3QJxkt1V0KbRRbrkoYX/tTwQj96zSSCAtTL4oMmKi5jc88Z'
    'TD+5gokmJJBvJhJZf4pNmopvxPHRyV94eC41fOCNUfNpnqe5771KyS+yCi7fSkRuSYrSkhwpSxONVQNvatW5yqpQAVDyNNnAGHxs6mqOkavyAgYwI6u6ZA23'
    't/DrslX2ipZhvcGpY73RuJLs19eafhOcRGoVyxxQVJDiwdtaX1W5XBHmNNoGFzDWj998kxaB5WgmPCIcvn9+/vznF6fh09P3z5+cnntzrPyp2aIbtaUd8r3X'
    'L8/CV+9ehm//9ub08dNzD9Nfn52++vnF4/P+/Zf/9aJzy9kErH0BkpdghlyAFHLhsSwhlOVdTifONmGwVWQK8CITLtM0LnarkeyoUd6LdAWQ/qdR1xLAmWvC'
    'Rsl4DfsghcFBCGmtQ7Ed4ynULNb6V4xo9FekVQ5oqR3e8G18FZpOE+/ywltXhIoFhGhhwIpzqwsMIqEbkApW12p1E6ZVmVWlf+Fd6ZJ0d/DEM+hgV5ziHuYf'
    'MGrSgL9Bpd7lTJBjLN4i7sA5gBKZXQsu0Sz3J6w3xqgR0btst8UY/HsZV7W5v5CFsAKxWmoVJWT+OeSDsUewN9lorLFDsn/WWSsrZu8XMdLrNT87UHe6PFil'
    'UUdY1pWRlkmzHZA67bYcCjCcXFXySr3E1PipIgr5oeX6kEIgLu9++P5wvQqXslCH1qQOwCWjanMjS7MqlqU6gNbW+u4QpuPVqkVAg8Sx7xsGDkUiN9gAXXAA'
    'AFqSy9A94zMgZ4yNhcEVkFfG3nTa13wXap5RZpB07daGb3cvHgr1q1pVgISDAxbhoN4k7AunFfVOWH9ipq1Hfch1qULaOplEox41s9qds8kj2CDUgBuK5+YW'
    'u1ysC57TgtYphbKlalOliIOsJAREzIJxQjigaJVQunNVJRG5ZKaNY8YK8ihOkBrns+zTMr7rftvyOk3qaEhuJ5dwu6rERpBNHay8WQf9vTYvgP+kSIJmIt6E'
    'kTGWh8734O+knloCnwI4MpdbxImZaC5OLqe9BVqjnhoLd3Q2vayDAvu9SQyRRO3QPe9Vo/o6TalvABnnvaDBu/EqTVSzEU/avLRODWd1smgSRMCCKnXEKeOa'
    'Uk3FCSO2Rjw7f4uxv1Xq/n9bFCTMgdc17JDfmSWC8q5sxwTGujh2e0fiGP/+Ko6Ckwf/ndDVifhBHAe4oK9H9C/4K65OxHd4+COujmnkd/iCD2vCdTI8o9w6'
    'U9E4G+Et36H0x5uNDqDYXfLzUYyiTat923Vfu58kkf1aMzS95O1Y4P8Qp3rkh043Gwg23UNuLHPz7bQvEsfiDwMtxVNOsTaw+AWoeD3BzKUlf9m19DacDzk1'
    'Zl4nsqRBuSp5lXh36O5Y7lm1jDWqIVne/7HBF4YOkZa5JJNF8MG1SXAYD5Ehr1OTbBYVHK6EktwCqhfFO1DWjeiW2SZSEgWVdCL6zB3PQqXDiI9QYdYSi0Vn'
    'NYQH+0An7oO6XKAB7ngSicfXA/bF7NfMLXzZUEjxucQ1gh8VbnB1xHSNqI1QEmkO7XWAAL8uM4jBRdnJtM1iz+AKp/zMrLj2zlVbYULj9cJMgFaYi48O4U92'
    'OWT4elNtRJ2f9NXOz9OwTalDJ4MJr/TSu0S67h+LR4/Ed0eNCCPVy6N6rf0Rt1eps5+hkmrKdBNq7/9Vylg20XVGyRAudtTmtawy0WuoIKQKsElZyB/rJwEV'
    'wV4H5TCObgZxKqPC75Bwy6JG8sZpLzxT26OAvJYnD76Hopos0Fb9zZOOQ8PuWhpwqOw6zBBdKZ1tKaxxYyOLkB0OD5xJwZUC1sNDkwI50Ia3eyTRcc0VtmOa'
    'C6XsdltqJWA/9Bruz12WohK3yOiXyM2QEDm+bZRgilvorVfu+q2+3774hYG/0ZoZceFxZbHNlBHVMzHEI+nqEVCdO6KgDEbmUWcMWw/ZLw052Sf4qYUsshaD'
    'MGxJBWnhPS9OaRM2/P53WgZFJ8Nhm1kzpM9sUTesjYBRY5WnmTsEnWBzg0jpW4BhEJ8ZDw5TN/r8h8McPF6b6ur4xBT7h6ALFh+2eV8rAHW+To5m5CaF7ZyZ'
    '8tTUpndgruSQPrZbVN1zC6AMC/1PJb5F9vBNo3V6yEb6rfjL0Y/f2zDJ+f5OeDHPZci9r91wYppNAUbdhJDgSvlD3U2Dda4UYKYR4tt6+f2I8yoV13ILHWXE'
    'liBwE3UoE5bhtMFWypdjaSkzyjDvtT1kOXU0vPfWX2i0DX9UodkaziJcqZHput3FooXwACnAOq6KazeFQeaeUa31sW0KtBVOvUvIKD6keVRwHtjt0JgmVbuz'
    'XDKRj7R4Q8UT9vCCHmF/aoDq9W76frQmR0KGV+WF60rXwHK5WWoYI6IPkazDTiuOWaljcKaSM7bGo6mdNNIp9OEri6EhzIQpFxcdawu9qUDQLWsqc5ePK+ov'
    '2vSjGdFyOlYnzGrXt9MZA5wpJrME0bFMc+hbDZ0hPDpmxT3xnNp+xq6MjVIUPHv3kLxciiY8U/RsQL2OnmNmxTroZquW2z35r5HRKCRirB/Jh/eKRLGyodCi'
    'x0x0b+YlcjqyUb+PMzMXeXjYZ6y0n1uwbd7/wdW9gUwkfaRWcwG34ncNgWuzBohQJG+5zdBuZOtzna10XLEhwj5GbkyGDsfwCExDhm5vLhytuY5Qgy7c2zrl'
    'vPHn7oRPw/LgS5CinVaPnhkX5cyXfTWAJW86ySibQZWT2+134c6umwmtRM6NDQFzmBRmy3nZdlxz2YyaD2QdjfEWj+oNj6qcoJhw3LGHhxT6ubFsSh5q7yju'
    'Lrfh3orAO3jhaM+BynE8HR/7RQZL/RhgakPANP5spk+2275uqtdtywmmy1KQwXkmIQzrBGkuTmAEvTx1LnansGOlp2d2PazyuDPXuQ1f7Way850pLnXnWGdz'
    'o+XRFXvJLfkQ54Rz4dU5GHcXQmQ4V4mKyNSBb0u51LEut6hhqgQeg/HPZFyo0YLakazJ+vue1ykHpp8jYwQd0XJjJLVnOibyOaK2gzzf31+etYpp9947Dr4P'
    'Hng7V7DtvNC86urL/oXtvzqzNR8EiR2ypuLarTqOnjkxWyezj87L6HG++mk+587gMwNkvNEYZlJmr9jIOObaw/RZzINPo8bUVigY1gSfFg7dMNYtFKdud42r'
    'xKgCTPrG4WZc4Sfl4mSKVNRr+mf00wVkp3E3G++fDXtmPegbRI0qQXJ/40/7aQ6KVhrQ6Ye6mesvnFJYM0Le5nV6NN3swe2iUn0M8jt6qObditOs7g5o2k0v'
    'NZAOpsM5tnn9bl6stUUpw/SKsuc1N6bu/8V1f/3ClZvXfOeXN++adhO7XrfBNNIwchOV3V0+pmV0xu8Bix3VjWkjsOD76zrGoLqtSG3xvv4YulFkGIQj9YSs'
    'm/qdtCQLMSU6sG4Aal6m5E2I+HGj8jAviibVOCJPKVPqKuJeSAhJK1ADsAHseixf9XORdgmjCYwzX2atqwyAw1Bq8aDnU5/qWsdmlrEkfCXjxAUBR7BJk7RM'
    'kZz5dV0UQc2UHsCHqfFglG7TLxLajAji9MqbjT7FPH7alht2CqGm730wxYOzSvcBbrTuaBPp7ivGMx7/1e9NKn4T9w9+ZyWBKdfIymMA1fBlBx+CgEz29MKh'
    'NQ277UG29ag9DuZ1snD5en52OrPCLhz5FvjfXaR5y2mqgtG2Ros2pVthOVoJmAODmy5kfkR8IuNGzs14x95JUQ2iuQYzEikwk0+2NDFwWRW/VSqSHPbKa2pD'
    'tQ+dN9z8Ptd23Wltarx/GmL1kPsVSKheYvvhmk611OOyNI6BJ7pwuukdZfbV026lLKvm5fbansT42NDV0adDM2bPsYwBZXW3UlnJTeNXafksrZKIM8xxHoCf'
    'wPtKDXNrtuqEFvT3HNzoHCixAo2dD3m/eXN+PvfI0o+GPBsovNgBXoSJG3nnf2bUjHge0k7SD7O2Rz2CK7ORDvaACtUypJKfFhb1LgwMKubAIPR4S2vsBzpr'
    'OXrUUKw7ZAW18TcjBEdqIFd9A3wnzXkxSgYwSnH+Fl5FJ9AQeO0BGm9oPzG/cf8gDhpgdoQm5QE6rMTmRXJa9Kubr2MM3JSaOjFDZvhcxS4a42su4Sk3Y4SM'
    'TE6IgVgPxmmY/GjtPSEp5wKmu/hIu3/YbsU8OFl/Er/onx86Z5AWH9ttHRu7oyPj/nRCIFgewgmZbxErlflHwXFLxPr9z3l6o5IznakRr89kUTitCRQkcQ+b'
    '7GmLEQyMvgLZPg+hX8rxKOeW0S8F4HociosN5bZ9XkbFqCd9kEA8a/WLB92JVgAnxL41A0/vMqSSIxqrx91oYnn/om6WGFRZRIybMxl0WGLR9PCaWwjWscyw'
    'U/BL4HpULPpQ1/p0024fkgEgHbWcD5TT+CMfA2VPNl+Dc/4o/IMhzWlAhUjfVtv2x+eXQG4wRnfSGc+pjR3PKV0bNRUtUyyAsVksV8qbXhz8cPzjyfyyTwED'
    'awqU+30phY6eaIjmvt+F80KRguK0EzD7jHci++XEURWT+c/z16/MGStWms1GXe01ITLHlq/SOOTnjQb5ajrG5mVtDzFSV3t/Sq/RT0aI/5tR2lBHEmHeXvLA'
    'dgQlCN1tbaodWmKZprE/YrRg9UhQF57ga0/coSEDGRsR6SmdNKOnW8NeeuOZTeN7dtfMVBpdS3NkpTEHN0z/p3AyS3dsLXmbkTIbnJSODnSJOmoSP4mj/gJ7'
    'd+jRVyQvhnW/raD4wH26NG9YeNLnOiPEDFUQBR1Ql8VKa1P/7uqWmLD7tSSGMdUGsaH97H05uOYWLPch+L1wef97+y5cPjQHPGUuPjYq+fMOlfz5U/cEoGHE'
    'tlA2UifDA95npgljz+CZP1LYHqbOXzk0TRJzIDmSdPQGWVOu7EuhpgHCh1apNq6P/QeP86uKKrczfkL9IK4cwfUiDKN0FYZTZ2YgoyiUdorvHRyY080wOWq8'
    'LupziVykLswhcjOiOMQEVG4oulJqSanD7nl6WR9LSgM6ze9N967aOVyJxTGPGCanhi2U0Iq3d749CnVgysuvn8+9rgPb6/p/zDdm4mrNDMcYglw7iz9oXuF3'
    'jFcmW9+n20Hvrw/4Xn3Mi2Wz9zq9ObdUsCsp857heRJRL62n35noKUykoqcC59TScD2DnHTbescgG+ty0aPdvr2s9dYcZ0YlD22RvQSFXCte0CjGGN3oyRya'
    'ZaAzViv4oLSHZ/hVE+Ps3pMihSmXTFs0bw8mdYjN+e+NekrpbFa7Rue2cyykntbZT7e9tOs0356taKcPu7Uzd486GHUEeAK5MKRELQw5HoUhgVUYWmUZRZ1v'
    'qTg9vdP0SoSgbDr5P1BLAwQUAAAACAAAACEAtDCZ50gOAADiJgAAIQAAAHNjcmlwdHMvcHJlcGFyZV9zcGVlY2hfdHJpZ3JhbS5weZ1aW2/bRhZ+16+YcoGI'
    'TCU6TtGiUOJiXcdJg3UbI06yWLgGMSJHEhuKw3JI22qaH7SPxb7ta/7Yfmcu5FBSnGT9YGluZ8585z6jIAjOa1HxmjNRsGXNF5LJRZ7mvGAblsp1lRecqZbV'
    'bdnka8FOzl8znquCZ3LCqlpc56qRTBM4kQWfx6NRtWlWsmQqrfOqUQdL0SQZb3hcbdh0St+UaJiqhEhXycMHD79B7yIvhGIFL5ctX4qfZSaKuOF1vPxjm1ql'
    'uRWJXd/UOZheG9p2bLqm9WjP27zIppbz0egXyUTZ1KLkrMwZJuU4qD43umVdSpYJ8MCevXwdswtZYPptU3PBXp09ixeqASA3ss5U3Nw28SgIgtFoUcs1S5JF'
    '27TgKWH5upJ1w3hZyoY3uSzVaOT66iWYU8KsSWXZgHqRz90a27PmJRCo3aoVVytMcs3flCzdd6kMqYo3K4/OOZoTdg5+zqXKb6npVqhV2+RF12rnVS1ToZTr'
    'AeIkCEO2EeuKWo7uK0GfvN48yWuRAq+Nnbep8nLpZh2Xmwl73oiaY0Z39g1fF6PRyxcvXrEjzWAIzEA7SaK4FkoW1yKMYpJd2ajLw6vRaPT3LTwysWBrXlUi'
    'S+Z5mWHLUMm2Bvsze2TIdV6IbMbmUhYRm/7Q8XH5iyzF1WzE8AexHX/4i0PSkLsShT4Ke1GJ8ikkDK23Ol9D2mCt4S00OxOqaj/8mzSULVowKZiE4uUlL2LS'
    'AyKcLxiE3nGh++hvk4si61q1gKaUuklyAxz2FOyABWe+/j8RKT7qA6u+B0rU12jefv/dgTGJg2KdZGZSnKaGCccV6BJ5gMuB16YRKoz0BIIUgx33hkAYtM1i'
    '+n1g5qAP59fqi6njv+VlWrQALCDtUQeFXMarYKynWlHQNGquQW8RBob1X9p1tQkm7J7XjB6NHVjESZxKnC70NozYV0fsEPz543YXM9YjW8MPCfaGF604rWtZ'
    'h8Fp0XHknNiKs5Sv5zn81SMICCJnvCqs3UPj0pXQZg89gKTjYACT5gFOpeCp8LmcDDD6mo1/LTuUHsNTAPb1D36fajJxm4qqQe+v5ZPTk7Pjl6cJJpIeY0Gi'
    'BUV+Cwh9nAV7uEl3Smw9Noha6C9EAz1+CWqkRED/8iqEJUG1wcJsZnZk9/RO7F0HJeQRUh8hHBAvAbt3j3U9MLzAn675W9XyxlDNy2te5FkCF9euYR0QxG0F'
    'QEXGiBZJU1N41BF43317enb87CIZQIBT04eZ/d6usuf7Z5034oQOhGN+9HjGpiZszxBMuYHmG13rT6TntGX+eyuSqqlJiLMZtjBfLprsuE5/+IECZLUK9w7O'
    'ZoS6dUqRd1bC9iu9MPJBs3adCKO6JzpmGLwMDfb04pWPmV4lrXYx2TZVS6bTHWdihSEVprUN+9NrQ1/guT1i7ggazxeVDlVMmk+fqLfEjsYQ9ZIcQ1O3oh89'
    'OGCaFst0eCg2M4N+3aYNAe9kFlokbmRbZPC3hUw5VnEcfiVqeNeiMCjHQwA1w47IFvB6Y6J9X6+cWHAmjuXoDt3dL4YbfRR9Uihxp297NdhsFqeFVIhiW3I3'
    'g58heL38ozvCCmDo1jHUm94Hai+vuU3IU4T0j0IhPC5QPxr4dfrro9GC/H+xn5SJGC5IRAjHFH6zfAnFCGmeibk6yELEXWQ94UXaFtq1UuJCjrUtGTnZ/BpJ'
    'IvzAvJAwMQVdzUs45nrJa8qzSvby+OculN7kiIyaHYm4HAb1HN6HK2ZU3wsAOpi6HCnWGYVl0kydsECt+MNvvwuieCVu7Zg7j0sldboYIqFMZzhj2lxi8YQy'
    'mauJ5R0+YXhoSii6U78RdZdJdmdt5xRkMok0wmaRH/5bometswcIG9ynba3whZJVm8fyDgNYBUekhZ1RzhcXkmcqDHUKhUwhk6k6sKm0OvBy6QPkS/kCp4xp'
    'WRCZDECrRmS0gFLgDciW1Adhr9kCctFftEj0tpeBzsiDK9JiGrsMSrCO9hHCwb483eoYphOQl4EFIrH4X1Ec0VtfOpFc3RXJzzi5j0W+RFBO8w//KSl25xko'
    '7EDtIv0QRpfKdJ7MZZ6GPS1zGVx1XHszY3GLogYZU8+gQ3VLHD75AwqTn8De7uXmOZTEXShR9OTlJhw4MKvJw/1JQjpH6ulbKV7S0JVHk0ROfSTyMLD1DcJp'
    '0BU4ns/wcNgvrH9BHgQZzLhX6pxYa/ANcispqS4EWo8oAaPAgGyWl6Zu1EUkyTj2Nq0QrpFIPNMV6YZbYzUGdW0NLpOzYOJLLtpyDU64vhXHyOkhEHwkKv9D'
    '+IijaVyfQd2C7C+OvlCNT3s19dVTJ6JtJUzV3aDnw190sIufjqeg6VRaqD16bMukeP0WkTa0NdPRKwTjiZFCIt/qZmTdPFXWR+zd+9617pZyIWgd7W6ivW7j'
    'ZvcHBYRLk/Zrk+pmRNoKyLSC7bmW315CmhNbcRo/7wMNVaz/JK9CHNiBoRqSCq/Feo6kIS/3T9F6xOuGzj8oiEOzMNYWE+spOwuhNgWY0oMRe8wekkro1uX0'
    'oTbUoR8MvPHDK10L3mFcu4zSH9W8edmKnUFtq0c9+X3c0o72XLkiVMOIeWauFWH/tnfqLVRVsyVKHcx1FJP2pqgWy5ZK5X2ud4s7c/cQwy28TVoF0EKrGFG8'
    'qAVKJMe7Nsqv2bfsPgsP2ePH7JsHH0FrL9uGMWKazz1PA7/xLXuW/8iQKNS9XQ00xXicU4rWG1Ea5+IpysTnENI0nmLCFkWrVp7F+X9aya1uxjoPSBstGkPK'
    'ZDW2WHGAOE9uk58bm/zAVJai2Y+EBTeV1Yaoy/lvoaNqlu1yptXBxAXyDoGZnxi+gtnw4L5rnA1g2MuO9+e85Ix1idngkO/9uIhkJtR8aTf7br/pvP9UNHrK'
    'iwahZWCcB+4KT2719xd6nkZ40f5dAD+z5k2CmKPgGHGQQ7CzFbdn2zGBIDOAtnXRD1Pjaj9mNlTPjGCwXpcySbVCpkndJm1ZoLXmKtFaHlz16Hm47qQhXnmg'
    'k5esXVcqdLMmcA6UVx09jGB2wa9lcFf94Lw53aKuRbgTe60RvXKB7o5QbZJwfUeb2JpobxK+m3Cf2Gtp4V33uIsvWDw7N1fG38SHDxBUYfEIcemKHcaH6KJi'
    'Qza1dJe+XcbdXtNtnLGkm1WersKgve7TWgznyuOjV76Xhnlf/TD7EYObIxbhl8VcyrfmfnzCUP8AcgCJExQaIZ38QPC/8axzn5TvED9a7Db5SUkBje93t4Y2'
    'AtMEaIe+mkT24t2q+rO/MHWwMcUt3s2K+xvkGAIML4Nlro0VtWxJd0+BvthvRH2Esm8+K7veUk7TlUjfojoO9CsCmMc/hGeIJJdkPxC/u+CN0NTTt5zsx7af'
    'ngTD9cRSvx32z0TD01W3NbLTNZZe7dnH5Ka6evC207MSewVz575YP9XX/jThp9PjJ3Q2MkWzR0zXUlXY4d1tB/835O3uYknZa2nlEnH6msoaalBJMm/GjdIb'
    'csMs3Kncl+CZ5YuFwRIbNtPUXDZ2J9yG0T7iDEow0+dMn5SWz6G0beO0VpTXeS1LqpiTWkpyx2aN02Pz4euqHf+0qrbX4PYaO5hDmHWdQtjmlR5TQmQWgG2O'
    '9mmmN4fix/37EqZj+kDu5PWT4+TN84vnP56dJk9O3zw/Ob2Adw+IjZOfj/9xmvz4+vnZk+T8+OXx2dnpWXJ2+ub0jPw/4a9xW+WFVElatYBsbywJzo9f/WSX'
    'DAAjBy/JCzQrBe+iG5a1S7PIRpS9aFV5RVwa11Vs4dbvFekBiim3U/TyRizpFSBoS8UXYjpHCJgioJL57Q+E09rSc5cdtfi9RaFCiKqpTTPdq57Jpq8oZF0f'
    'edDvysU8HlmXGiMnCgN76W+j6cTefEf6joMurPuKaf/L08T236FnQ1yQ4LRVrF9FOhw/kg/gL73Jjr78dSj4DCzA4lzc5dK2xJl6sgq8p0jYS8EbypDg1CjG'
    'TlhJzzwT1r9NPWJ9LRiY9MDLQt6NzTbjmaMUW79gE64wmow16TFd4Yf6a5y40SSJJh75sd59PNMf/qzJ2IpuPPNezdwj5ICEFbVOv9R4tuKKN9i4XzYZDx9Z'
    'xkMW6KoDWX6is656P4HBK8Y4eh9FwT4V7uOEiUX2CX5wB6VF2ccPM+Uy0DiZ+yTjNXRH72ypMuwmO3fXRx3X84m7DceRfTPNbLXYhR1GGY5iVgxCuWezjHcV'
    'WO6edRmKNO3Dw46vgSjAHs3oBoc4wxf+P7zqB7rN8C24fw0kzhyfVKT+8SU2sw56n2kWD0L/jridl6rfitqvyGwCMOtyA/JTVowzdyqzVWL3mVlutz3LAFC6'
    'ZoGKIPbodxr45Gp6fRh4AhEFUKSEd4eOzUcpj0/MDzOS7Tpv+Mq/E6gGTn1n8ed7fhOvdgKzXmzfWXSF8alaiFD//ErIVjkvvd/EdPeTVOdYNbAlzprncGNU'
    'xWBVV8S4n90o+qEJvQeQYgJsnbqhM0/zBuTUI7jUMuXmsYRlnK5Uy5xVgm5ZS9Hi2Oam0JYxOtskDXI/OImP7bPsuR6hmk2LDMI7SpJMppCPtzLmmf+SO52a'
    'e3icqtlU4sj81gLH4m3R6FYYmBnqwPy6BsmJLIu8FAfD3+jwmH4GEkR3bmYLa3+3O+cPfvKDVfAIdK5AwZIhZxhacOf6wU+E7lqPRcrcwBEZ/UGEVDiolULq'
    'igcPS/r+nnoHlW7kv7tpkvY58DnCFKS9dbLJ9q+ZmGR8PZedGyUnD/YI4ZjyrISig+HGSGf3MULfi29z67G18zo2MfOtiOhcfjbvl6jd8w/c8nCzAQj9Zru3'
    'AK780i98D2BJoJAkdOmQJDpBSxKyqyQJDBXj8y82KH7WpyhJQmN10eh/UEsDBBQAAAAIAAAAIQBeWrR1iwcAAPgTAAAZAAAAc2NyaXB0cy90cmlncmFtX3dv'
    'cmtlci5wea1Y3W7bOBa+91NwdSV1XTVp08xMWi/QnU0KLDJtkXTmJhMItETFbGRSQ1Jp3MAPs8+wd3PbF5vvkPqxY6UNsGsgiHTI88NzvvNDRVH00fA5/8QL'
    'bdi/z9+/YzU3nImKVcusELkuhGG6lLnk1SsmPom8cdywXCtmG/Zh5RZ4epHu77GfP/yaTibvNJPLWhvnZdRGr0TuNFMSez9qky/Y8/Rlyk65ZUvujMyFZTcS'
    '+hVkLjUrK83di+esks5VYiJUIbG0aoXVUrO5BE1ddUYxXlcy56zSV5nVpVvyW2yvOKuF4pX8wnP59b+KzSuurtNJFEWTSWn0kmVZ2bjGiCxrDWZcKe24k1rZ'
    'yaSlzbkVhwfd2yerVeCuuVtUct6xfsBrt6muuCu1WXbvRgxPVjcmFwyHx2FwdJt1tG6PXdmgwa1qOmVLfqNWvU2qWdYrkqHqjuTIs93LELjJZFKIkvTWWhVx'
    'zVfwbnHECpm7C+vMlOReJuzpP9g7rcTRhOEHFx0vpROsUZx9/VORc0lCI6zjr+BnRE4oyz8hcgruugGhQHg6sNwgXpxZh2eTkr9JaG2kcjH5Ly1gvo3vIn0d'
    'HbGPphFT9uRJa9p6ynhV6c+Z4mp2wisrkikg0djFjHYm7YGWvK5Fkc1XTtiYQnHkI+APAj39OX7Wyxpsc2BRsVOpmttgPq8F/hnBKxFsvzK81FPw5lUjC6zZ'
    'Bgv4z43hK9sfwwEgFZuxPf+GKAOnSoDP64+jZxTWZ1ZU5TMosVGSQkmROXHr4iS1QKojBhsnwUYvRYqqsBBKK2FPDBD7h9nLpN8nS1YJFYftCfvbjB0OQuiH'
    'nHQ4o9iQDAa+FJBtILmZxyb6/ff4Yu/pD5d3L9ZJNEWeLOcFp1TM4cR8YWKKk3+92L+csh8TCoBXefHycsuYXjosAZZ8HFKCc3Uj4iT5jm3AknFTxKWAda2C'
    'vcv2+NHTaFAVfP73GUU2xv4p2z9EpP1rKwUEv90IZLQKHD1UpIpHgQFOAY68MVZvoYBqW6PYPw1Yz2sh8sW/WmgjvVhpUBIYzwF83uOiy7C7qPYVEcjuykAa'
    'KNmNMBalJYY/I5+u2ENu889p1q1nGW3wKY4Nqt5cmW65lEVtGsDsemGxe8EtdxA5FACIOhfuxLozwPAXUKJkHVyFOvuZQHe37g4QStOM6dqXQDxRTfCrvnhm'
    'vqC61ebCZgagcqXIeqmGyDuz2oaBEX9QGYEIXwoo5W1MApKtbboWxhdij1zPchHpOrrc2gUMbmycsUgq6aJthQFA5GErXAa3Zm5BKWnjXm5LiC6THc6lJljM'
    'QnL3DJ46tj0vr7YN9o68Z3VwA4oCiRm2p1cCuO8XkJr+JdrVIssNfnQs8n3gJCZgIkp2feC1cgnk/sarRhwbo00cAREayEddyZE4fEgCyKXU0ErnKIfp96wg'
    '55NitNDCm/QoJD7WyuOKmQblY0nHZU4K4M1qanVi03gY8ICh/4M1A0u6zRD3p99VeT9dgIuLaIs4gokh7TZ0hsLzPizFoxZ64egXmS9JIkLRDiSp7pPmgi/7'
    'F5Qnh+kra4kPy+a5biy2ZjZHvxyE+fPYa1n7lLILXRWDcKGu3GI4bUtWcyB9TNmuDzcK0o4/ztq1mOpnyNFnLPp4+jYtIZ7qZxT+ttY/a1PY1N22O0ay3ZdT'
    'KNwaL3blP4B9gWmlHQw2fw8nSkyZ0ipFHW2fXrMRlSis3E8QLrPyi0D3O9j76fCbGXQWUqbNofcIBdBLCbTwqQKTNOvTHaN3XQmnXzHre+KCf/Hze4lpbM7z'
    'awx0Z29+GUuvofMNtetoOPF0u0sFn2JDII6jzv8iWHidGUzIHcv9iZkqpmksvxLxztLZr+dv3h5n58enJxjBmgz5AVHsCdvfe36wHqnc98cTH7lCgIrZxGxW'
    'dWqc32tEfiwZ6UQeCy2upfVtlCK/oQi13Hfmx9bGtz58FmxofnQdwoUH97VeZO7vdUBeGFuMqIV7oKJ7vReDLZfbqbc7DsXdWaZd8foGPB7r9O4WMdu159s+'
    'hyYx5vPO/jNaj/8PFu5ornE9GVFsF8iyTeR4wkjlbwd7v+zn+ue+HMie9BoXjscC4gQjJ/dtUV9JZynhb77+p5IFH4v5DXFSz8GgSbfOeVOWCGy49abzw4Pg'
    'vWH2CVKpoIMVQp0IF7MpK3BhFbPodXngbz3e9PYAaa7r1YjrcXAqgdAtbUnDm4iDQUmKuhM/ekA47Y/qpWg7dtSdLvKOZuy4HwiC5ilZ80UYbeN4fxpiiItQ'
    'fz7yU/hCAdJWS9/WKKodmJBto0DpzDrxG7xxuPmP+Iu6wrezgnrQroIdj5XUD0z7ZaQQth30+BG7621e3/PhUOhzNC8feqrKF3eRhS1C5aHu26ZyaUdBvzKy'
    'xpXn4Tq/+dscNbQhed7XcSuVt/RHi0N4rhr0h3FxAEQQt/bXmEClAtzFI1Di5HKjMojbXNSOHft/FFlumSCfbnv9oc8d/pMGeqJnIYOiO4JV7N8T3PXoQp1l'
    'awTCk9bR+t4HkBAKf83dwxUXKOuYPMiyjC68WdbCLET+fGWdWB7f+g8LdB1OJn8BUEsDBBQAAAAIAAAAIQAAAAAAAgAAAAAAAAAPAAAAc3JjL19faW5pdF9f'
    'LnB5AwBQSwMEFAAAAAgAAAAhABPP2yZGAAAATQAAABkAAABzcmMvaW5mZXJlbmNlL19faW5pdF9fLnB5HYrBDYAwDMT+TBHdAMzCCqE9RKSSVimbMQKLEfGz'
    'ZQPYgkNDi72PC8X8YNCLqVTKqXtTmeYpg17/MFOa0O/89LKEvgJYPlBLAwQUAAAACAAAACEAqd42TgIDAABUBwAAFAAAAHNyYy9pbmZlcmVuY2UvZnN0LnB5'
    'rVTNbtNAEL7nKYY9ILsqVlq1kQgNEqrKqRIgoV6iKJqs1+kIe9fZXZcWxMPwCIhH6Isx678kJqI94EvWm88z33wz8wkhLnGlpLIIH0ql3zs/BdKuVFLS428N'
    'KI3HFMGRBol2jRaU4xvjQBOglcYlQojRKLOmgOUyq3xl1XIJVJTGekCtOYAno12LKdHf5rTqAB/5ddSenbeV9A3MP5Sk1x3qnX4YjUapysAqTJeZ88tbPigb'
    'hXDTOkoMr95CStLPOc5x+GQxHQE/zO9aKVD3Mq8c3WGhtFeQIxfUlr4ijZYQcvI+Z6ROCTWkqtMETpJJcl4XGgJ+JX9b15EY/j8SdiViQBf4KyyapOGhrL1K'
    'AuvoLIYXs7bIpET5JRIXJI7h9OT0fHL+ejKexNtvw2ORnIIbzCt1Za2xkbjKWQBZWWdAG/CktIJK71TSEZamYIK0ylUi4j6o5todzGC+6K8yY4G7pTmZXqvo'
    'dEDB0Te1TNEjzAa17MG40pyV6NF1pWf7oQ5XNBw/YHm05JHb5d0xgV6+Su8IuE07Hy+GvHj+4AQuZk0A/p08i9dnKk3PibVm2Y02klIzJHYXvhrIE3IdVKgG'
    'N3PAkP8qUN3cBEuGpU2eJGXSqYoEOkkk4i3eI+XDjo7j3cENZAOq6eT4qcF8kmVNyB1qIH3abDbcxTpbjW1Wm7Fhm6NvVEaRuFPWsYswTmQ5rl04lJb3z/IW'
    '1G9sS9a3B99csT85ER+3ydv6ubgmwbxDLuACxsCL0N3X39W327r/rvl6Z+94QljsHPmYszd6LPDxFx9SlcPaYtZPjVVskBq+i+BhbHJKTJvO8eg2jPdvT/j2'
    '6Kgh9qP3wE1F7LIFNxt5xYMfPtMIr+5preCS7diHLrGXP/7UKbs65qQ5iXnDfgI3Snpj28nHtCD2y2D7PBWS6WPvhH2nDtlyL3fU6drXvAhTJWSgIQbC7wI4'
    'ZSCXir3pY3xY6j5qNxoLmM141XfCNZOygJdwFsdPtLItDVjURmBlg7f+S6tBT5usoz9QSwMEFAAAAAgAAAAhAGcQ4lrCAAAA9gAAACAAAAByZXF1aXJlbWVu'
    'dHMtdHJpZ3JhbS1ydW50aW1lLnR4dB2MQW7DIBBF95wCqcvK42C7adJqVr1ApaoHQDCJUcgMAhzHty/uZvTn6f/3on8kiibWFNutkln01/ev9u335MSHS3DW'
    'S9bSQrDxU7dG4FJttHkfwoP4Aarr6Fmz7QJ7enZLjnquNZWPvveychTrIW3N72aQfO3XOfYuLeqfIBowI5jXnfByT9tOhgkmVbcU+Lq7iUsQLogTmDc4qEJ1'
    'SVUkNnQ8wQCDWmeiiHiAycCoUkiITdLEyt3tjRBHGN7hrP4AUEsDBBQAAAAIAAAAIQArkxfkkgIAAHYEAAA1AAAAY29uZmlncy9lbnRyZW5vc19vbmxpbmUv'
    'c3BlZWNoX3RyaWdyYW1hX2NvbXBhY3RvLnlhbWyNU8tu1TAQ3ecrLLEkyvXNvXldVm1paaGliPIQIGQ5jpNYTewQO4XyW+zY9sc4btLHBoTkiZVz5oxnxuMn'
    '5EzZ3hA3qmbkPSfXpLz5bb9NsuLPyCiHUVqpHRfq5pcmB0Zbd2Qd4Z3SklfevZPCTSMnPR+ioDeV7MyOVNzxVcd1M/FGslvUruwgpWjZchQTph+4cCbgo2jV'
    'lWG25XGS7khWpttqk4p0m6VJJShPZF7QsqRCZEWSJpnI46zMqs16W5d1Iis45VmWl9U6TatMBsO1a41m46Sd6uVfslnYVan0ahYEziCTB9k6Wm+i9VMxTMFS'
    'JEPiNcpDrUGrOmMZyB2JAy8wk2NWNpOujN2RlNKgnOZG7gICyQ+GatUVAmcUJCCl76H4Fikl73FuHvmfjjunhGQzOGNcmMkCZVbwDjIa5V6G2i6ZvVQDcy0u'
    'rDVd5dOn99wgNe/ctRekBfqWrXOaJEWxTTb+JKkb1z528kpdSusQJqiNlj23d8V/2dsLyd4h7Bh2DvsI+xSS/ZAcAHqOhe3Q21sYmKOQvAjJMZATb0BeYn8V'
    'ktOQnIXkNRb4c8Q5B/cmJNBdYMHpHRa2997AfwgJvnD6jAXs4uT0a7BMEobUaN9rgwmT+r+mcL4G1RtWKTsYrcpOMgz9wMfbcKxRJS5zCz//EsYr7h2FmfHE'
    'N0/1ykk2WgwDHxu+KB4JrLIODZyJGMTdtMz+DzNT0ODR0bvbEIOxCnOp0PjWucHuVqtGuXYqI1S9qr+rrpPOLVXtH5xEYCEEibSQYRzzIqWJEDXN13kl65yW'
    'aV3lGMEsKXKZFXRLcwlJPeGhS/vPtuEAeM7PZUfwOmi09k1YnvCiXVoc03hzH+fMh4kcH6PmZ/AHUEsDBBQAAAAIAAAAIQChHwn23QEAAD4DAAATAAAAZnVl'
    'bnRlc19zaGEyNTYuanNvbm2STW4bMQyF9zlF4HUTixJJib3MgKKodNB47MxM0AZF714ZMdCmiNbC+/h+ft3d3x82W+fLvh3tfLqo7dN2cbdv077OT6ueHi9v'
    'h6/3By0KsRVCLlIaGUVkDp5JrZkigHKL0CISOjtALGaZE6WkiUvBw5d/UZfVL7r656g4xJJxjC1T6ExJhEviHjRDFucUmjMWUpHaS0g5h6wNE7FLLZI/om7a'
    '04/z+t3XGwLR+xAqnC2oF/VItRg6DJgn1FBMc/PxtHoWEMDM42uhqrmS3BCrHadpXuZ9mm7CnmowxCilGxigaK8drQwLvUrEmNURHBmlSsKRnJAI1Fwo1kL0'
    'V3heuq++mP+PSE21Z+jDb2ADSN4Ia/ZqCUHGnVEySA01G3FCio0RSLhdr8KgnyH6tt/Ug0lyDhGsjKyJCROLtY4dxLQTjHC4k8ecc00hg0NtXTmHJq2Dv6uv'
    '/vI6r37yZd8ebg08rK/LPp/8cf+5X0ndsdQaUkWIZApGGclFSqKCBIolZB5/Ru6JWiqQgaEDWI563cc7yc5Ln5+24yANK+dtOi/P8+LHj8vS6Tbu8+Obnp6v'
    'eOokxlc/lWPqEEKQxDDq7VTAYx032eheS88xplYZKpi1McReRpWHu99/AFBLAQIUAxQAAAAIAAAAIQComcqMFhIAAMI1AAAhAAAAAAAAAAAAAACkgQAAAABz'
    'Y3JpcHRzL2NvbXBhY3Rfc3BlZWNoX3RyaWdyYW0ucHlQSwECFAMUAAAACAAAACEAtDCZ50gOAADiJgAAIQAAAAAAAAAAAAAApIFVEgAAc2NyaXB0cy9wcmVw'
    'YXJlX3NwZWVjaF90cmlncmFtLnB5UEsBAhQDFAAAAAgAAAAhAF5atHWLBwAA+BMAABkAAAAAAAAAAAAAAKSB3CAAAHNjcmlwdHMvdHJpZ3JhbV93b3JrZXIu'
    'cHlQSwECFAMUAAAACAAAACEAAAAAAAIAAAAAAAAADwAAAAAAAAAAAAAApIGeKAAAc3JjL19faW5pdF9fLnB5UEsBAhQDFAAAAAgAAAAhABPP2yZGAAAATQAA'
    'ABkAAAAAAAAAAAAAAKSBzSgAAHNyYy9pbmZlcmVuY2UvX19pbml0X18ucHlQSwECFAMUAAAACAAAACEAqd42TgIDAABUBwAAFAAAAAAAAAAAAAAApIFKKQAA'
    'c3JjL2luZmVyZW5jZS9mc3QucHlQSwECFAMUAAAACAAAACEAZxDiWsIAAAD2AAAAIAAAAAAAAAAAAAAApIF+LAAAcmVxdWlyZW1lbnRzLXRyaWdyYW0tcnVu'
    'dGltZS50eHRQSwECFAMUAAAACAAAACEAK5MX5JICAAB2BAAANQAAAAAAAAAAAAAApIF+LQAAY29uZmlncy9lbnRyZW5vc19vbmxpbmUvc3BlZWNoX3RyaWdy'
    'YW1hX2NvbXBhY3RvLnlhbWxQSwECFAMUAAAACAAAACEAoR8J9t0BAAA+AwAAEwAAAAAAAAAAAAAApIFjMAAAZnVlbnRlc19zaGEyNTYuanNvblBLBQYAAAAA'
    'CQAJAJ0CAABxMgAAAAA='
)
source_bytes = base64.b64decode(SOURCE_B64)
if hashlib.sha256(source_bytes).hexdigest() != SOURCE_SHA256:
    raise ValueError("El paquete de fuentes no supera su SHA-256.")
workspace_compact = previous_project.parent / ("tic_proyecto_trigrama_compacto_" + uuid4().hex[:8])
workspace_compact.mkdir()
with ZipFile(io.BytesIO(source_bytes)) as archive:
    for member in archive.infolist():
        relative = PurePosixPath(member.filename)
        if relative.is_absolute() or ".." in relative.parts or member.is_dir():
            raise ValueError("Ruta inesperada en el paquete de fuentes.")
        target = workspace_compact.joinpath(*relative.parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(member))
for name, expected in json.loads((workspace_compact / "fuentes_sha256.json").read_text()).items():
    if hashlib.sha256((workspace_compact / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Fuente distinta del paquete: {name}")

spec_compact = yaml.safe_load((workspace_compact / "configs/entrenos_online/speech_trigrama_compacto.yaml").read_text())
spec_compact["modelo"] = str(source_model.parent / "speech_trigram_compacto")
spec_compact["compactacion"]["origen"] = str(source_model)
spec_compact["python_runtime"] = str(source_runtime.absolute())
spec_compact["preparacion"]["fuentes"] = str(source_tree.resolve())
for key in ("archivo_sha256", "fonemas_grafo", "busqueda"):
    if spec_compact[key] != language_cfg[key]:
        raise ValueError(f"El perfil cambiaría {key}; revisar antes de convertir.")
for line in Path("/proc/meminfo").read_text().splitlines():
    if line.startswith("MemAvailable:"):
        available_gib = int(line.split()[1]) / (1 << 20)
        break
minimum = spec_compact["compactacion"]["minimo_disponible_preparacion_gib"]
if available_gib < minimum:
    raise RuntimeError(f"Esta preparación necesita {minimum} GiB disponibles; ahora hay {available_gib:.1f}. Usa RAM alta.")
if Path(spec_compact["modelo"]).exists():
    raise FileExistsError("El destino compacto ya existe; se conserva. Revisa su manifest antes de repetir.")

COMPACT_OUTPUT = Path(RESULTS) / ("compactacion_" + uuid4().hex[:8])
COMPACT_OUTPUT.mkdir()
(COMPACT_OUTPUT / "fuentes_compactacion.zip").write_bytes(source_bytes)
config_path = COMPACT_OUTPUT / "trigrama_compacto.yaml"
config_path.write_text(yaml.safe_dump(spec_compact, allow_unicode=True, sort_keys=False))
(COMPACT_OUTPUT / "preparacion.json").write_text(json.dumps({"source_sha256": SOURCE_SHA256,
    "scope": "conversion_y_carga; pendiente_inferencia_cpu_con_gru"}, indent=2))
print("Modelo original:", source_model)
print("Modelo compacto:", spec_compact["modelo"])
print("RAM disponible para preparar:", round(available_gib, 1), "GiB")
print("Fuentes y registro:", COMPACT_OUTPUT)
print("La conversión conserva el grafo y puede aumentar su tamaño en disco; la carga usará map.")
env_compact = {**os.environ, "CUDA_VISIBLE_DEVICES": ""}
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    env_compact[name] = str(spec_compact["hilos_cpu"])
command = [str(PYTHON), "scripts/compact_speech_trigram.py", "--config", str(config_path),
           "--build-runtime", "--prepare-model", "--check-loading", "--output", str(COMPACT_OUTPUT / "carga")]
with (COMPACT_OUTPUT / "preparacion.log").open("w") as log:
    process_compact = subprocess.Popen(command, cwd=workspace_compact, env=env_compact,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process_compact.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        returncode_compact = process_compact.wait()
    finally:
        process_compact.stdout.close()
        if process_compact.poll() is None:
            process_compact.terminate()
            try:
                process_compact.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process_compact.kill()
                process_compact.wait()
if returncode_compact:
    raise RuntimeError(f"Preparación interrumpida (código {returncode_compact}). Revisa {COMPACT_OUTPUT / 'preparacion.log'}")
report_compact = json.loads((COMPACT_OUTPUT / "carga/comprobacion_carga.json").read_text())
print("Carga map verificada:", report_compact["verified"])
print("Pico de RAM al cargar solo el trigrama:", report_compact["peak_worker_rss_bytes"] / (1 << 30), "GiB")
print("Pendiente: comprobar memoria y velocidad de GRU + búsqueda en CPU local.")
print("Resultados:", COMPACT_OUTPUT)


Modelo original: /content/drive/MyDrive/speech_bci/modelos_lenguaje/speech_trigram
Modelo compacto: /content/drive/MyDrive/speech_bci/modelos_lenguaje/speech_trigram_compacto
RAM disponible para preparar: 81.5 GiB
Fuentes y registro: /content/drive/MyDrive/speech_bci/experimentos/trigrama_t12_20261101_210724_101948c3/compactacion_c134c197
La conversión conserva el grafo y puede aumentar su tamaño en disco; la carga usará map.
Using Python 3.10.15 environment at: /content/tic_proyecto_trigrama_colab/data/language_models/runtime
Audited 7 packages in 104ms
running install
/content/tic_proyecto_trigrama_colab/data/language_models/runtime/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.


In [14]:
#@title Comprobar GRU + trigrama compacto en CPU (tres frases de val)
# Copia esta celda completa al final de la sesión del notebook 13.
from pathlib import Path, PurePosixPath
import base64
import hashlib
import io
import json
import os
import subprocess
from uuid import uuid4
from zipfile import ZipFile
import google.colab
import yaml

if not all(name in globals() for name in ("PROJECT", "RESULTS", "PYTHON", "CHECKPOINT_PATH", "COMPACT_OUTPUT")):
    raise RuntimeError("Se necesita la sesión del notebook 13 donde terminó la compactación.")
compaction_output = Path(COMPACT_OUTPUT)
loading_report = json.loads((compaction_output / "carga/comprobacion_carga.json").read_text())
if not loading_report["verified"]:
    raise RuntimeError("La carga map anterior no está verificada.")
previous_config = yaml.safe_load((compaction_output / "trigrama_compacto.yaml").read_text())
model_manifest = Path(previous_config["modelo"]) / "manifest.json"
if hashlib.sha256(model_manifest.read_bytes()).hexdigest() != loading_report["manifest_sha256"]:
    raise ValueError("El manifiesto del modelo compacto ha cambiado desde su comprobación.")
SOURCE_SHA256 = '7b0f988b3f01d9c646d60f07cc0b2194ad8b192dc05773a90c66a093bf7254e1'
SOURCE_B64 = (
    '...'
)
source_bytes = base64.b64decode(SOURCE_B64)
if hashlib.sha256(source_bytes).hexdigest() != SOURCE_SHA256:
    raise ValueError("El paquete de fuentes no supera su SHA-256.")
CPU_PROJECT = Path(PROJECT).parent / ("tic_proyecto_compacto_cpu_" + uuid4().hex[:8])
CPU_PROJECT.mkdir()
with ZipFile(io.BytesIO(source_bytes)) as archive:
    for member in archive.infolist():
        relative = PurePosixPath(member.filename)
        if relative.is_absolute() or ".." in relative.parts or member.is_dir():
            raise ValueError("Ruta no válida en el paquete.")
        target = CPU_PROJECT.joinpath(*relative.parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(member))
snapshot = json.loads((CPU_PROJECT / "source_snapshot.json").read_text())
for name, expected in snapshot["files"].items():
    if hashlib.sha256((CPU_PROJECT / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Fuente distinta del paquete: {name}")
(CPU_PROJECT / "data").symlink_to((Path(PROJECT) / "data").resolve(), target_is_directory=True)
profile = yaml.safe_load((CPU_PROJECT / "configs/entrenos_online/speech_trigrama_compacto.yaml").read_text())
for key in ("busqueda", "fonemas_grafo", "archivo_sha256"):
    if previous_config[key] != profile[key]:
        raise ValueError(f"La comprobación cambiaría {key}.")
previous_config["comprobacion_cpu"] = profile["comprobacion_cpu"]
CPU_OUTPUT = Path(RESULTS) / ("compacto_cpu_" + uuid4().hex[:8])
CPU_OUTPUT.mkdir()
(CPU_OUTPUT / "fuentes_cpu.zip").write_bytes(source_bytes)
config_path = CPU_OUTPUT / "trigrama_compacto_cpu.yaml"
config_path.write_text(yaml.safe_dump(previous_config, allow_unicode=True, sort_keys=False))
print("CPU, CUDA oculta, hilos:", previous_config["hilos_cpu"])
print("Límites:", previous_config["comprobacion_cpu"])
print("Resultados:", CPU_OUTPUT)
environment_cpu = {**os.environ, "CUDA_VISIBLE_DEVICES": ""}
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    environment_cpu[name] = str(previous_config["hilos_cpu"])
command = [str(PYTHON), "scripts/check_speech_trigram_cpu.py", "--config", str(config_path),
           "--checkpoint", str(Path(CHECKPOINT_PATH).absolute()), "--output", str(CPU_OUTPUT / "comprobacion")]
with (CPU_OUTPUT / "supervisor.log").open("w") as log:
    child_cpu = subprocess.Popen(command, cwd=CPU_PROJECT, env=environment_cpu,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in child_cpu.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        cpu_returncode = child_cpu.wait()
    finally:
        child_cpu.stdout.close()
        if child_cpu.poll() is None:
            child_cpu.terminate()
            try:
                child_cpu.wait(timeout=10)
            except subprocess.TimeoutExpired:
                child_cpu.kill()
                child_cpu.wait()
if cpu_returncode:
    raise RuntimeError(f"Comprobación CPU detenida (código {cpu_returncode}); revisar {CPU_OUTPUT}")
CPU_REPORT = json.loads((CPU_OUTPUT / "comprobacion/comprobacion_cpu.json").read_text())
print("Comprobación del subconjunto:", CPU_REPORT["verified"])
print("Pico conjunto de RSS:", CPU_REPORT["peak_tree_rss_bytes"] / (1 << 30), "GiB")
print("RTF observado en esta CPU:", CPU_REPORT["inference"]["timing"]["rtf"])
print("Queda pendiente repetirlo en la máquina de la demo.")


CPU, CUDA oculta, hilos: 2
Límites: {'max_frases': 3, 'bloque_bins': 4, 'limite_rss_total_gib': 6, 'reserva_sistema_gib': 2, 'timeout_segundos': 900, 'periodo_muestreo_segundos': 0.1}
Resultados: /content/drive/MyDrive/speech_bci/experimentos/trigrama_t12_20261101_210724_101948c3/compacto_cpu_7c577fad
Pipeline CPU: RSS total=0.38 GiB; disponible=81.32 GiB
Pipeline CPU: RSS total=0.39 GiB; disponible=81.29 GiB
Pipeline CPU: RSS total=0.47 GiB; disponible=81.23 GiB
Pipeline CPU: RSS total=0.47 GiB; disponible=81.22 GiB
Pipeline CPU: RSS total=0.65 GiB; disponible=81.04 GiB
Pipeline CPU: RSS total=0.65 GiB; disponible=81.02 GiB
Pipeline CPU: RSS total=0.65 GiB; disponible=81.02 GiB
Pipeline CPU: RSS total=0.65 GiB; disponible=81.01 GiB
Pipeline CPU: RSS total=0.65 GiB; disponible=81.00 GiB
Pipeline CPU: RSS total=0.65 GiB; disponible=81.00 GiB
Pipeline CPU: RSS total=0.65 GiB; disponible=81.00 GiB
Pipeline CPU: RSS total=0.65 GiB; disponible=81.00 GiB
Pipeline CPU: RSS total=0.65 GiB; dis

RuntimeError: Comprobación CPU detenida (código 1); revisar /content/drive/MyDrive/speech_bci/experimentos/trigrama_t12_20261101_210724_101948c3/compacto_cpu_7c577fad

In [15]:
#@title Repetir comprobación CPU con control de caché del grafo
# Copia esta celda completa al final de la sesión del notebook 13.
from pathlib import Path, PurePosixPath
import base64
import hashlib
import io
import json
import os
import subprocess
from uuid import uuid4
from zipfile import ZipFile
import google.colab
import yaml

if not all(name in globals() for name in ("PROJECT", "RESULTS", "PYTHON", "CHECKPOINT_PATH", "COMPACT_OUTPUT")):
    raise RuntimeError("Se necesita la sesión del notebook 13 donde terminó la compactación.")
compaction_output = Path(COMPACT_OUTPUT)
loading_report = json.loads((compaction_output / "carga/comprobacion_carga.json").read_text())
if not loading_report["verified"]:
    raise RuntimeError("La carga map anterior no está verificada.")
previous_config = yaml.safe_load((compaction_output / "trigrama_compacto.yaml").read_text())
model_manifest = Path(previous_config["modelo"]) / "manifest.json"
if hashlib.sha256(model_manifest.read_bytes()).hexdigest() != loading_report["manifest_sha256"]:
    raise ValueError("El manifiesto del modelo compacto ha cambiado desde su comprobación.")
SOURCE_SHA256 = 'edaa3402fbfec00d217a539979bc30ffaea3b1bb39c7a0e95b03d4e6f8fdd60c'
SOURCE_B64 = (
    '...'
    'FAAAhPsBAAAA'
)
source_bytes = base64.b64decode(SOURCE_B64)
if hashlib.sha256(source_bytes).hexdigest() != SOURCE_SHA256:
    raise ValueError("El paquete de fuentes no supera su SHA-256.")
CPU_PROJECT = Path(PROJECT).parent / ("tic_proyecto_compacto_cpu_" + uuid4().hex[:8])
CPU_PROJECT.mkdir()
with ZipFile(io.BytesIO(source_bytes)) as archive:
    for member in archive.infolist():
        relative = PurePosixPath(member.filename)
        if relative.is_absolute() or ".." in relative.parts or member.is_dir():
            raise ValueError("Ruta no válida en el paquete.")
        target = CPU_PROJECT.joinpath(*relative.parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(member))
snapshot = json.loads((CPU_PROJECT / "source_snapshot.json").read_text())
for name, expected in snapshot["files"].items():
    if hashlib.sha256((CPU_PROJECT / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Fuente distinta del paquete: {name}")
(CPU_PROJECT / "data").symlink_to((Path(PROJECT) / "data").resolve(), target_is_directory=True)
profile = yaml.safe_load((CPU_PROJECT / "configs/entrenos_online/speech_trigrama_compacto.yaml").read_text())
for key in ("busqueda", "fonemas_grafo", "archivo_sha256"):
    if previous_config[key] != profile[key]:
        raise ValueError(f"La comprobación cambiaría {key}.")
previous_config["comprobacion_cpu"] = profile["comprobacion_cpu"]
previous_config["cache_mapeo"] = profile["cache_mapeo"]
CPU_OUTPUT = Path(RESULTS) / ("compacto_cpu_" + uuid4().hex[:8])
CPU_OUTPUT.mkdir()
(CPU_OUTPUT / "fuentes_cpu.zip").write_bytes(source_bytes)
config_path = CPU_OUTPUT / "trigrama_compacto_cpu.yaml"
config_path.write_text(yaml.safe_dump(previous_config, allow_unicode=True, sort_keys=False))
print("CPU, CUDA oculta, hilos:", previous_config["hilos_cpu"])
print("Límites:", previous_config["comprobacion_cpu"])
print("Caché del grafo:", previous_config["cache_mapeo"])
print("Resultados:", CPU_OUTPUT)
environment_cpu = {**os.environ, "CUDA_VISIBLE_DEVICES": ""}
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    environment_cpu[name] = str(previous_config["hilos_cpu"])
command = [str(PYTHON), "scripts/check_speech_trigram_cpu.py", "--config", str(config_path),
           "--checkpoint", str(Path(CHECKPOINT_PATH).absolute()), "--output", str(CPU_OUTPUT / "comprobacion")]
with (CPU_OUTPUT / "supervisor.log").open("w") as log:
    child_cpu = subprocess.Popen(command, cwd=CPU_PROJECT, env=environment_cpu,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in child_cpu.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        cpu_returncode = child_cpu.wait()
    finally:
        child_cpu.stdout.close()
        if child_cpu.poll() is None:
            child_cpu.terminate()
            try:
                child_cpu.wait(timeout=10)
            except subprocess.TimeoutExpired:
                child_cpu.kill()
                child_cpu.wait()
if cpu_returncode:
    raise RuntimeError(f"Comprobación CPU detenida (código {cpu_returncode}); revisar {CPU_OUTPUT}")
CPU_REPORT = json.loads((CPU_OUTPUT / "comprobacion/comprobacion_cpu.json").read_text())
print("Comprobación del subconjunto:", CPU_REPORT["verified"])
print("Pico conjunto de RSS:", CPU_REPORT["peak_tree_rss_bytes"] / (1 << 30), "GiB")
print("RTF observado en esta CPU:", CPU_REPORT["inference"]["timing"]["rtf"])
print("Queda pendiente repetirlo en la máquina de la demo.")


CPU, CUDA oculta, hilos: 2
Límites: {'max_frases': 3, 'bloque_bins': 4, 'limite_rss_total_gib': 6, 'reserva_sistema_gib': 2, 'timeout_segundos': 900, 'periodo_muestreo_segundos': 0.1}
Caché del grafo: {'rss_archivos_umbral_gib': 1.5, 'liberar_al_final_frase': True}
Resultados: /content/drive/MyDrive/speech_bci/experimentos/trigrama_t12_20261101_210724_101948c3/compacto_cpu_96640ff8
Pipeline CPU: RSS total=0.47 GiB; disponible=81.23 GiB
Pipeline CPU: RSS total=0.47 GiB; disponible=81.23 GiB
Pipeline CPU: RSS total=0.66 GiB; disponible=81.04 GiB
Pipeline CPU: RSS total=0.66 GiB; disponible=81.03 GiB
Pipeline CPU: RSS total=0.66 GiB; disponible=81.03 GiB
Pipeline CPU: RSS total=0.66 GiB; disponible=81.03 GiB
Pipeline CPU: RSS total=0.66 GiB; disponible=81.03 GiB
Pipeline CPU: RSS total=0.66 GiB; disponible=81.02 GiB
Pipeline CPU: RSS total=0.66 GiB; disponible=81.01 GiB
Pipeline CPU: RSS total=0.66 GiB; disponible=81.01 GiB
Pipeline CPU: RSS total=0.66 GiB; disponible=81.04 GiB
Pipeline C

In [16]:
#@title Guardar el runtime CPU antes de cerrar Colab
# Ejecuta esta celda en la misma sesión, después de la comprobación CPU correcta.
import hashlib
import json
import os
from pathlib import Path
import re
import stat
import subprocess
import tarfile
import time
from uuid import uuid4

if "CPU_OUTPUT" not in globals() or "RESULTS" not in globals():
    raise RuntimeError("Falta la sesión anterior: conserva Colab abierto y ejecuta primero la comprobación CPU.")

cpu_output_backup = Path(CPU_OUTPUT)
report_path_backup = cpu_output_backup / "comprobacion/comprobacion_cpu.json"
report_backup = json.loads(report_path_backup.read_text())
if not report_backup.get("verified") or report_backup.get("returncode") != 0:
    raise RuntimeError("La última comprobación CPU no terminó correctamente; no se publica un respaldo válido.")
spec_backup = report_backup["language_config"]
runtime_backup = Path(spec_backup["python_runtime"]).absolute()
if not runtime_backup.is_file():
    raise FileNotFoundError(f"El runtime temporal ya no está disponible: {runtime_backup}")

probe_code_backup = r'''
import json, os, platform, re, sys
from pathlib import Path
import numpy, torch, lm_decoder
native = set()
for line in Path("/proc/self/maps").read_text().splitlines():
    fields = line.split(maxsplit=5)
    if len(fields) != 6 or not fields[5].startswith("/"):
        continue
    name = re.sub(r"\\([0-7]{3})", lambda match: chr(int(match.group(1), 8)), fields[5])
    path = Path(name)
    if path.is_file():
        with path.open("rb") as stream:
            if stream.read(4) == b"\x7fELF":
                native.add(str(path.resolve()))
print(json.dumps({"python": platform.python_version(), "torch": str(torch.__version__),
    "numpy": numpy.__version__, "cuda_build": torch.version.cuda,
    "mapped_graphs": hasattr(lm_decoder, "SetFstReadMode"),
    "compact_writer": hasattr(lm_decoder, "WriteConstFst"),
    "prefix": str(Path(sys.prefix).resolve()), "base_prefix": str(Path(sys.base_prefix).resolve()),
    "binding": str(Path(lm_decoder.__file__).resolve()),
    "executable": str(Path(sys.executable).resolve()), "native_files": sorted(native),
    "platform": platform.platform(), "libc": platform.libc_ver(), "machine": platform.machine()}))
'''
environment_backup = {**os.environ, "CUDA_VISIBLE_DEVICES": "", "PYTHONDONTWRITEBYTECODE": "1"}
for name_backup in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    environment_backup[name_backup] = str(spec_backup["hilos_cpu"])
probe_backup = json.loads(subprocess.check_output(
    [str(runtime_backup), "-c", probe_code_backup], env=environment_backup, text=True,
    timeout=90))
if (probe_backup["torch"] != spec_backup["torch_runtime"] or probe_backup["cuda_build"] is not None
        or not probe_backup["mapped_graphs"] or not probe_backup["compact_writer"]):
    raise RuntimeError("El runtime disponible no coincide con el runtime CPU que hay que conservar.")
expected_runtime_backup = report_backup["inference"]["worker_runtime"]
for key_backup in ("python", "torch", "numpy"):
    if probe_backup[key_backup] != expected_runtime_backup[key_backup]:
        raise RuntimeError(f"El runtime cambió desde la comprobación CPU: {key_backup}")

prefix_backup = Path(probe_backup["prefix"])
base_backup = Path(probe_backup["base_prefix"])
if prefix_backup != runtime_backup.parent.parent.resolve() or prefix_backup == base_backup:
    raise RuntimeError("Se esperaba el entorno aislado del trigrama; se conserva la sesión para revisar sus rutas.")
for root_backup in (prefix_backup, base_backup):
    if len(root_backup.parts) < 4 or not root_backup.is_dir():
        raise RuntimeError(f"Raíz de Python inesperada; se evita copiar una carpeta general: {root_backup}")
marker_backup = prefix_backup / "runtime_build.json"
if not marker_backup.is_file():
    raise FileNotFoundError("Falta runtime_build.json del runtime compilado.")
model_backup = Path(spec_backup["modelo"])
manifest_bytes_backup = (model_backup / "manifest.json").read_bytes()
if hashlib.sha256(manifest_bytes_backup).hexdigest() != report_backup["inference"]["language_manifest_sha256"]:
    raise RuntimeError("El manifiesto del grafo cambió desde la comprobación CPU.")

# Inventario limitado al Python aislado y a las bibliotecas realmente cargadas.
# El grafo grande, los datos y los pesos se conservan en sus carpetas de Drive.
entries_backup = {}
for root_backup in (base_backup, prefix_backup):
    for path_backup in (root_backup, *sorted(root_backup.rglob("*"))):
        mode_backup = path_backup.lstat().st_mode
        if stat.S_ISREG(mode_backup) or stat.S_ISLNK(mode_backup):
            entries_backup["rootfs/" + path_backup.as_posix().lstrip("/")] = path_backup
for name_backup in (probe_backup["binding"], *probe_backup["native_files"]):
    path_backup = Path(name_backup)
    if not path_backup.is_file():
        raise FileNotFoundError(f"Falta una biblioteca cargada: {path_backup}")
    entries_backup["rootfs/" + path_backup.as_posix().lstrip("/")] = path_backup
# Conservar también módulos nativos que OpenFst o Torch podrían cargar después.
source_native_backup = Path(spec_backup["preparacion"]["fuentes"]) / "LanguageModelDecoder/runtime"
if not source_native_backup.is_dir():
    raise FileNotFoundError(f"Faltan las bibliotecas compiladas originales: {source_native_backup}")
for path_backup in sorted(source_native_backup.rglob("*.so*")):
    if not path_backup.is_file():
        continue
    with path_backup.open("rb") as source_backup:
        if source_backup.read(4) != b"\x7fELF":
            continue
    for native_path_backup in (path_backup.absolute(), path_backup.resolve()):
        entries_backup["rootfs/" + native_path_backup.as_posix().lstrip("/")] = native_path_backup
for name_backup in ("fuentes_cpu.zip", "trigrama_compacto_cpu.yaml",
                    "comprobacion/comprobacion_cpu.json", "comprobacion/inferencia_cpu.json"):
    path_backup = cpu_output_backup / name_backup
    if not path_backup.is_file():
        raise FileNotFoundError(f"Falta un registro de la comprobación: {path_backup}")
    entries_backup["evidence/" + name_backup] = path_backup
entries_backup["evidence/model_manifest.json"] = model_backup / "manifest.json"

BACKUP_OUTPUT = Path(RESULTS) / ("respaldo_runtime_" + uuid4().hex[:8])
BACKUP_OUTPUT.mkdir(exist_ok=False)
archive_backup = BACKUP_OUTPUT / "runtime_trigrama_cpu.tar.gz"
partial_backup = archive_backup.with_suffix(archive_backup.suffix + ".partial")
size_backup = sum(path.lstat().st_size for path in entries_backup.values() if not path.is_symlink())
print("Runtime CPU:", probe_backup["python"], probe_backup["torch"], flush=True)
print("Archivos:", len(entries_backup), "· Tamaño antes de comprimir:", round(size_backup / (1 << 30), 2), "GiB", flush=True)
print("Destino:", BACKUP_OUTPUT, flush=True)
print("Guardando copia. El grafo grande ya está en Drive y no se duplica aquí.", flush=True)
records_backup = {}
started_backup = time.monotonic()
try:
    with tarfile.open(partial_backup, "w:gz", compresslevel=1, dereference=False) as archive_stream_backup:
        for index_backup, (name_backup, path_backup) in enumerate(sorted(entries_backup.items()), 1):
            info_backup = archive_stream_backup.gettarinfo(str(path_backup), arcname=name_backup)
            if path_backup.is_symlink():
                if not info_backup.issym():
                    raise RuntimeError(f"El tipo del enlace cambió: {path_backup}")
                records_backup[name_backup] = {"type": "symlink", "target": os.readlink(path_backup)}
                archive_stream_backup.addfile(info_backup)
            else:
                # Guardar cada archivo como regular evita enlaces duros entre árboles de Python.
                info_backup.type = tarfile.REGTYPE
                info_backup.linkname = ""
                info_backup.size = path_backup.stat().st_size
                with path_backup.open("rb") as source_backup:
                    digest_backup = hashlib.file_digest(source_backup, "sha256").hexdigest()
                    source_backup.seek(0)
                    archive_stream_backup.addfile(info_backup, source_backup)
                records_backup[name_backup] = {"type": "file", "size_bytes": info_backup.size, "sha256": digest_backup}
            if index_backup % 2000 == 0:
                print(f"Guardados {index_backup}/{len(entries_backup)} archivos", flush=True)
    print("Copia escrita. Verificando el contenido comprimido…", flush=True)
    seen_backup = set()
    with tarfile.open(partial_backup, "r|gz") as archive_stream_backup:
        for member_backup in archive_stream_backup:
            if member_backup.name not in records_backup or member_backup.name in seen_backup:
                raise RuntimeError("El respaldo contiene una entrada inesperada o repetida.")
            record_backup = records_backup[member_backup.name]
            if record_backup["type"] == "symlink":
                if not member_backup.issym() or member_backup.linkname != record_backup["target"]:
                    raise RuntimeError(f"Enlace distinto en el respaldo: {member_backup.name}")
            else:
                if not member_backup.isfile() or member_backup.size != record_backup["size_bytes"]:
                    raise RuntimeError(f"Archivo distinto en el respaldo: {member_backup.name}")
                with archive_stream_backup.extractfile(member_backup) as source_backup:
                    actual_backup = hashlib.file_digest(source_backup, "sha256").hexdigest()
                if actual_backup != record_backup["sha256"]:
                    raise RuntimeError(f"SHA-256 distinto en el respaldo: {member_backup.name}")
            seen_backup.add(member_backup.name)
    if seen_backup != set(records_backup):
        raise RuntimeError("El respaldo está incompleto.")
    with partial_backup.open("rb") as source_backup:
        archive_sha256_backup = hashlib.file_digest(source_backup, "sha256").hexdigest()
    partial_backup.rename(archive_backup)
    inventory_backup = {"format_version": 1, "scope": "runtime_cpu_backup; requiere adaptación y prueba local",
        "verified": True, "local_demo_verified": False, "probe": probe_backup,
        "archive": archive_backup.name, "archive_sha256": archive_sha256_backup,
        "archive_size_bytes": archive_backup.stat().st_size, "files": records_backup,
        "model_directory": str(model_backup), "evaluation_directory": str(cpu_output_backup),
        "native_build_directory": str(source_native_backup),
        "elapsed_seconds": time.monotonic() - started_backup,
        "note": "rootfs conserva rutas y enlaces originales; no extraer sobre / ni activar este entorno sin adaptarlo."}
    (BACKUP_OUTPUT / "respaldo_verificado.json").write_text(json.dumps(inventory_backup, indent=2) + "\n")
    (BACKUP_OUTPUT / (archive_backup.name + ".sha256")).write_text(archive_sha256_backup + "  " + archive_backup.name + "\n")
except BaseException:
    print("Respaldo sin confirmar. Conserva Colab abierto; revisar:", BACKUP_OUTPUT, flush=True)
    raise
print("RESPALDO_VERIFICADO: True", flush=True)
print("Paquete:", archive_backup, flush=True)
print("Tamaño comprimido:", round(archive_backup.stat().st_size / (1 << 30), 2), "GiB", flush=True)
print("SHA-256:", archive_sha256_backup, flush=True)
print("Cuando aparezca RESPALDO_VERIFICADO: True, ya puedes cerrar Colab.", flush=True)
print("Para seguir en local: descargar esta carpeta de respaldo y la carpeta del grafo:", model_backup, flush=True)


Runtime CPU: 3.10.15 1.13.1+cpu
Archivos: 20833 · Tamaño antes de comprimir: 1.56 GiB
Destino: /content/drive/MyDrive/speech_bci/experimentos/trigrama_t12_20261101_210724_101948c3/respaldo_runtime_e8d287b5
Guardando copia. El grafo grande ya está en Drive y no se duplica aquí.
Guardados 2000/20833 archivos
Guardados 4000/20833 archivos
Guardados 6000/20833 archivos
Guardados 8000/20833 archivos
Guardados 10000/20833 archivos
Guardados 12000/20833 archivos
Guardados 14000/20833 archivos
Guardados 16000/20833 archivos
Guardados 18000/20833 archivos
Guardados 20000/20833 archivos
Copia escrita. Verificando el contenido comprimido…
RESPALDO_VERIFICADO: True
Paquete: /content/drive/MyDrive/speech_bci/experimentos/trigrama_t12_20261101_210724_101948c3/respaldo_runtime_e8d287b5/runtime_trigrama_cpu.tar.gz
Tamaño comprimido: 0.48 GiB
SHA-256: 1c91b2797c7a2837c6f237501e6ef95c1c1f8de603d8f84a3eaf0f59c9da51b9
Cuando aparezca RESPALDO_VERIFICADO: True, ya puedes cerrar Colab.
Para seguir en local: